# 11m — Busca de hiperparâmetros do SVM multiclasse (grade completa e validação cruzada de 5 dobras) e treino final

Este notebook faz com o **SVM (kernel RBF, 7 classes)** o mesmo que o notebook 07m fez com o RF: escolhe **C** e **gamma** por validação cruzada, mede a influência de cada um, escolhe a melhor combinação por uma regra fixada antes e treina o modelo final com o custo computacional medido. A avaliação no teste fica para o notebook 12m.

## O que é IGUAL ao RF multiclasse (mesma quantidade de dados)
| Item | Como |
|---|---|
| **Dados** | As **mesmas views** do RF (sem os atributos de AUC alto): **ajuste inteiro (239.671 linhas)**, validação (59.918) e refit (299.589). Os ausentes já estão tratados nelas (mediana do treino + indicadores `_ausente`) |
| **Dobras** | As **mesmas 5 dobras do RF** (mesmo código, mesma ordem das linhas, semente 42, estratificadas por classe). O notebook confere a **impressão digital** com a gravada pelo 07m (`7e97c47f`) e para se for diferente |
| **Pesos** | `class_weight="balanced"`, como no RF |
| **Métricas** | As do RF: ROC-AUC um-contra-resto (macro), acurácia balanceada, acurácia, F1 macro, MCC, **recall por classe**, validação **sem pesos e na proporção do treino**, **matriz de confusão**, dispersão entre 5 partições da validação e validação cruzada dobra a dobra |
| **Regra de escolha** | A mesma: ROC-AUC e MCC pela CV, **1 erro-padrão**, e entre as empatadas vence a **mais barata** (aqui, o menor número de **vetores de suporte**) |
| **Influência dos hiperparâmetros** | Efeito principal, "um de cada vez" a partir do padrão, interação (mapas de calor) e decomposição da variação |
| **Custo computacional** | Tempo, CPU e RAM de pico do treino, tamanho e vetores de suporte do modelo, latência (1 fluxo) e vazão (lote), e o refit cronometrado |
| **Arquivos** | Os mesmos CSVs, JSONs e gráficos do RF, e o registro de buscas |

## O que muda (e por quê)
| Item | SVM |
|---|---|
| **Normalização** | **Necessária**: padronização só das colunas numéricas (os indicadores `_ausente` ficam 0/1), **ajustada em cada treino**, sem vazamento nas dobras (a mesma do binário) |
| **Hiperparâmetros** | **C** ∈ {1, 10, 100} × **gamma** ∈ {0,25×, 1×, 4×} do valor `scale` = **9 configurações** (a grade do binário). O padrão do scikit-learn (C = 1, gamma = `scale`) está na grade. O `gamma` é o **valor absoluto** calculado uma vez no ajuste inteiro, e o mesmo valor vale no refit |
| **Sementes** | **Não há**: o SVM é determinístico (a mesma entrada dá exatamente o mesmo modelo). Cada configuração tem 5 treinos de dobra + 1 treino completo |
| **Escore** | A decisão **um-contra-resto** do SVM (`decision_function`, 7 colunas), não uma probabilidade; a classe prevista é o maior escore. O ROC-AUC por classe só usa a ordem dos escores |
| **Custo "barato"** | O número de **vetores de suporte** (define o tamanho do modelo e o custo de prever), no lugar dos nós das árvores |

## Paralelismo: só com memória suficiente
Os 54 treinos (9 configurações × 6) são independentes, e cada um roda **num processo à parte**. O notebook roda **vários ao mesmo tempo**, mas **só se houver memória livre**:
- **Antes de iniciar cada treino adicional**, confere a memória livre: ela precisa cobrir ~`RAM_POR_PROCESSO_GIB` × 1,3 **mais** `RAM_RESERVA_GIB` (1 GiB nunca é tocado). O valor por processo começa em 1 GiB e é **atualizado com o pico medido** nos treinos já feitos.
- **Espera ~10 s** entre uma partida e outra (a memória do treino anterior leva alguns segundos para subir).
- Se a memória livre **cair abaixo de um nível crítico** com vários treinos rodando, o **mais novo é interrompido e volta para a fila**, e o teto de simultâneos diminui.
- **Com um só treino rodando, nunca trava**: sempre inicia o próximo. Para rodar um treino, exige ao menos `RAM_MINIMA_UM_TREINO_GIB` (0,7 GiB) livres: se não houver, espera até 10 min e para com uma mensagem clara.
- O teto é `PROCESSOS_MAX` (padrão 4). Os dados ficam em disco e cada processo os lê por **memória mapeada** (o processo principal não copia nada).
- O notebook **mede o ganho real do paralelismo** (soma dos tempos de treino ÷ relógio) e o grava no `tempos_busca.csv`.

## Quanto tempo leva (estimativa por extrapolação da calibração; **é muito longo**)
A calibração mediu até 111.592 linhas (7 min 23 s). Com o ajuste inteiro (expoente do tempo de 2,1 a 2,3):

| Parte | Provável | Faixa |
|---|---|---|
| 1 treino de dobra (191.737 linhas) + prever a dobra | ~33 min | 31 a 38 min |
| 1 treino completo (239.671 linhas) + prever a validação | ~55 min | 49 a 63 min |
| 1 configuração (5 dobras + 1 completo) | ~3,7 h | 3,4 a 4,2 h |
| **Busca: as 9 configurações, 1 treino por vez** | **~35 h** | 33 a 40 h |
| Busca com 2 treinos simultâneos (se couberem e o ganho for ~1,8×) | ~19 h | |
| Busca com 4 treinos simultâneos (ganho de ~3×; exige ~6 GiB livres: 1 GiB de reserva + 4 × 1,3 GiB) | ~12 h | |
| Refit (299.589 linhas), 1 treino (+ 1 do padrão) | ~1,1 h cada | 1,0 a 1,5 h |
| Custos de inferência (latência e vazão) | ~20 a 40 min | |

**Cada treino é salvo ao terminar**: se for interrompido, **rode a célula de novo e ela retoma**. Deixe rodando em várias noites, com o computador na tomada e **sem suspensão**.

Para encurtar (com a perda indicada): `AVALIAR_VALIDACAO_POR_CONFIG = False` (só a escolhida é avaliada na validação; economiza ~8 h, mas as colunas de validação por configuração ficam vazias), `COMPARAR_COM_PADRAO = False` (sem o refit do padrão; economiza ~1,1 h, mas o teste não compara com o padrão), `REPETICOES_TEMPO = 1` (já é o padrão; o RF usou 3).

## Antes de rodar
Os notebooks 02, 03m, 04m, 05m e 07m precisam ter terminado (este usa as mesmas views, o mesmo refit e as dobras do 07m). **Feche o Meet, o Edge e os outros notebooks**: quanto mais memória livre, mais treinos simultâneos. O notebook só lê o servidor.


In [1]:
import os, re, json, time, http.client, urllib.parse
from pathlib import Path
import pandas as pd

pd.set_option("display.width", 200); pd.set_option("display.max_rows", 200); pd.set_option("display.max_colwidth", 120)

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()


def load_env(path: Path) -> dict:
    values = {}
    for line in path.read_text().splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, _, value = line.partition("=")
        values[key.strip()] = value.strip().strip('"').strip("'")
    return values


env_path = REPO_ROOT / ".env"
if not env_path.exists():
    raise FileNotFoundError(f"Arquivo .env não encontrado em: {env_path}")
env = {**load_env(env_path), **os.environ}

CH_HOST, CH_PORT = env["CLICKHOUSE_HOST"], env["CLICKHOUSE_PORT"]
CH_USER, CH_PASSWORD, CH_DATABASE = env["CLICKHOUSE_USER"], env["CLICKHOUSE_PASSWORD"], env["CLICKHOUSE_DATABASE"]
CH_SCHEME = env.get("CLICKHOUSE_SCHEME", "https")
HTTP_TIMEOUT = 1800
print(f"ClickHouse: {CH_HOST}:{CH_PORT}/{CH_DATABASE}  |  usuário: {CH_USER}")

ClickHouse: clickhouse.afonsolelis.tech:443/railway  |  usuário: clickhouse


In [2]:
# ---------------- transporte HTTP direto (conexão reutilizada + reconexão automática só para leituras) ----------------
class ErroConexao(Exception):
    pass


CONEXAO_ESPERAS = (0, 5, 15, 45, 90)
_conn = None
ULT_STATS = None                  # estatísticas (linhas lidas etc.) da última consulta JSON


def _fechar_conexao():
    global _conn
    try:
        if _conn is not None:
            _conn.close()
    except Exception:
        pass
    _conn = None


def _http(query: str) -> pd.DataFrame:
    """Executa via HTTP. Repetir leituras é seguro; escritas não são reenviadas aqui (quem repete é carregar(), com verificação de contagem)."""
    global _conn, ULT_STATS
    params = {"database": CH_DATABASE, "default_format": "JSONCompact", "wait_end_of_query": 1}
    primeiro = re.match(r"\s*(\w+)", query)
    verbo = primeiro.group(1).upper() if primeiro else ""
    if verbo == "ALTER":
        params["mutations_sync"] = 2          # espera o DELETE terminar antes de responder
    caminho = "/?" + urllib.parse.urlencode(params)
    cab = {"X-ClickHouse-User": CH_USER, "X-ClickHouse-Key": CH_PASSWORD, "Connection": "keep-alive"}
    somente_leitura = verbo not in {"INSERT", "CREATE", "DROP", "ALTER", "TRUNCATE", "RENAME", "OPTIMIZE", "DELETE", "UPDATE"}
    tentativas = CONEXAO_ESPERAS if somente_leitura else (0,)
    ultimo = None
    for espera in tentativas:
        if espera:
            print(f"  conexão instável ({type(ultimo).__name__}: {str(ultimo)[:160]}); nova tentativa em {espera}s...")
            time.sleep(espera)
        try:
            if _conn is None:
                cls = http.client.HTTPSConnection if CH_SCHEME == "https" else http.client.HTTPConnection
                _conn = cls(CH_HOST, int(CH_PORT), timeout=HTTP_TIMEOUT)
            _conn.request("POST", caminho, body=query.encode("utf-8"), headers=cab)
            resp = _conn.getresponse()
            corpo = resp.read()
        except (OSError, http.client.HTTPException) as e:
            ultimo = e
            _fechar_conexao()
            if not somente_leitura:
                raise ErroConexao(f"Falha de conexão durante escrita: {e!r}") from e
            continue
        if resp.status in (502, 503, 504) or (resp.status >= 500 and not corpo.startswith(b"Code:")):
            ultimo = Exception(f"HTTP {resp.status} [código ClickHouse: {resp.getheader('X-ClickHouse-Exception-Code')}] corpo={corpo[:300].decode('utf-8', 'replace')!r}")
            _fechar_conexao()
            if not somente_leitura:
                raise ErroConexao(f"HTTP {resp.status} durante escrita. Resposta: {corpo.decode('utf-8', 'replace')[:500]!r}")
            continue
        if resp.status != 200:
            raise Exception(corpo.decode("utf-8", "replace")[:2000])
        if not corpo.strip():
            return pd.DataFrame()
        dados = json.loads(corpo)
        ULT_STATS = dados.get("statistics")
        return pd.DataFrame(dados.get("data", []), columns=[m["name"] for m in dados.get("meta", [])])
    raise ErroConexao(f"Sem conexão com o ClickHouse após {len(tentativas)} tentativas: {ultimo!r}")

In [3]:
# ---------------- TRAVA: só é possível escrever em tabelas prep_* ----------------
PREFIXO_NOVO = "prep_"          # tabelas que ESTE notebook pode criar / alimentar / apagar

_LITERAL = re.compile(r"'[^'\\]*(?:\\.[^'\\]*)*'")      # literais entre aspas simples (aceita \' escapado)


def _sem_literais(s: str) -> str:
    # texto SQL sem o conteúdo dos literais: ';' ou 'DROP TABLE' dentro de aspas NÃO contam na checagem da trava
    return _LITERAL.sub("''", s)


_VERBOS_ESCRITA = ("CREATE", "INSERT", "DROP", "ALTER", "TRUNCATE", "RENAME", "OPTIMIZE")
_VERBOS_PROIBIDOS = ("DELETE", "UPDATE", "ATTACH", "DETACH", "EXCHANGE", "GRANT", "REVOKE", "KILL", "SYSTEM", "SET", "USE")
_ALVO = re.compile(r"\b(?:TABLE|INTO|VIEW)\s+(?:IF\s+(?:NOT\s+)?EXISTS\s+)?(`[^`]+`(?:\.`[^`]+`)?|[\w]+(?:\.[\w]+)?)", re.I)
_RENAME = re.compile(r"(`[^`]+`(?:\.`[^`]+`)?|[\w]+(?:\.[\w]+)?)\s+TO\s+(`[^`]+`(?:\.`[^`]+`)?|[\w]+(?:\.[\w]+)?)", re.I)


def _tabela_de(ident: str) -> str:
    # 'banco.tabela' ou '`tabela`' -> 'tabela' (e exige que o banco, se informado, seja o atual)
    partes = [p.strip("`") for p in re.findall(r"`[^`]+`|[^.]+", ident)]
    if len(partes) == 2 and partes[0] != CH_DATABASE:
        raise PermissionError(f"Escrita bloqueada: banco '{partes[0]}' diferente do atual '{CH_DATABASE}'.")
    return partes[-1]


def validar_escrita(sql: str, permitir_prefixos=()) -> None:
    # Levanta PermissionError se o comando escreveria fora das tabelas permitidas. Leituras passam sem restrição.
    s = sql.strip().rstrip(";").strip()
    s_chk = _sem_literais(s)
    if ";" in s_chk:
        raise PermissionError("Escrita bloqueada: envie uma instrução por vez.")
    verbo = (re.match(r"\s*(\w+)", s).group(1) if re.match(r"\s*(\w+)", s) else "").upper()
    if verbo in _VERBOS_PROIBIDOS:
        raise PermissionError(f"Comando '{verbo}' não é permitido neste notebook.")
    if verbo not in _VERBOS_ESCRITA:
        return                                                    # SELECT / SHOW / DESCRIBE / EXISTS ...
    prefixos = (PREFIXO_NOVO, *permitir_prefixos)
    if verbo == "RENAME":
        nomes = [_tabela_de(x) for par in _RENAME.findall(s_chk) for x in par]
    else:
        nomes = [_tabela_de(x) for x in _ALVO.findall(s_chk)]
    if not nomes:
        raise PermissionError(f"Escrita bloqueada: não identifiquei a tabela alvo em: {s[:80]}...")
    ruins = [n for n in nomes if not n.startswith(prefixos)]
    if ruins:
        raise PermissionError(f"Escrita bloqueada em {ruins}: só é permitido escrever em tabelas com prefixo {prefixos}.")


def executar(sql: str, permitir_prefixos=()) -> pd.DataFrame:
    validar_escrita(sql, permitir_prefixos)
    return _http(sql)


consultar = executar      # leituras também passam pela trava (e ela as libera)

# autoteste da trava (não toca no servidor)
_deve_passar = ["SELECT 1", "SHOW CREATE TABLE data2_portmap", f"CREATE TABLE {PREFIXO_NOVO}x AS data2_portmap",
                f"INSERT INTO {PREFIXO_NOVO}x SELECT * FROM data2_portmap LIMIT 1", f"DROP TABLE IF EXISTS `{PREFIXO_NOVO}x`",
                f"INSERT INTO {PREFIXO_NOVO}x VALUES ('idêntica a a; teste', 'DROP TABLE data_syn')"]
_deve_bloquear = ["INSERT INTO data2_portmap SELECT 1", "DROP TABLE data2_portmap", "DROP TABLE IF EXISTS data_tftp",
                  "TRUNCATE TABLE data_syn", "ALTER TABLE data_syn DROP COLUMN label", "CREATE TABLE outra AS data_syn",
                  "RENAME TABLE data_syn TO prep_syn", "CREATE DATABASE x", "DELETE FROM data_syn WHERE 1",
                  f"INSERT INTO {PREFIXO_NOVO}x SELECT 1; DROP TABLE data_syn", "SYSTEM DROP MARK CACHE", f"INSERT INTO {PREFIXO_NOVO}x VALUES ('a'); DROP TABLE data_syn"]
for _s in _deve_passar:
    validar_escrita(_s)
for _s in _deve_bloquear:
    try:
        validar_escrita(_s); raise AssertionError(f"a trava deixou passar: {_s}")
    except PermissionError:
        pass
print(f"Trava de segurança OK: {len(_deve_passar)} comandos permitidos e {len(_deve_bloquear)} bloqueados no autoteste.")

Trava de segurança OK: 6 comandos permitidos e 12 bloqueados no autoteste.


In [4]:
# ---------- ferramentas gerais ----------
import os, sys, json, math, time, zlib, platform, threading
from pathlib import Path
from datetime import datetime
import numpy as np
try:
    import psutil
except ImportError:
    psutil = None
    print("⚠ psutil não está instalado: o pico de memória e o hardware não serão registrados. Rode `pip install psutil` e execute esta célula de novo.")
try:
    import sklearn, joblib
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.model_selection import StratifiedKFold
    from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve
except ImportError as e:
    raise ImportError("Faltam bibliotecas: rode `pip install scikit-learn joblib psutil` e execute esta célula de novo.") from e

def lit(s): return "'" + str(s).replace("\\", "\\\\").replace("'", "\\'") + "'"
def q(c): return "`" + str(c).replace("`", "``") + "`"

def consultar_num(sql):
    df = consultar(sql)
    for c in df.columns:
        try: df[c] = pd.to_numeric(df[c])
        except (ValueError, TypeError): pass
    return df

def n_linhas(tab, onde=""):
    return int(consultar_num(f"SELECT count() AS n FROM {tab}" + (f" WHERE {onde}" if onde else "")).n[0])

def existe(tab):
    return int(consultar_num(f"SELECT count() AS n FROM system.tables WHERE database = currentDatabase() AND name = {lit(tab)}").n[0]) > 0

def colunas_view(view):
    return consultar(f"SELECT name FROM system.columns WHERE database = currentDatabase() AND `table` = {lit(view)} ORDER BY position")["name"].tolist()

def ler_view(view, onde=""):
    # Lê uma view inteira. O id_linha (UInt64) fica como texto, para não perder precisão; o resto vira número.
    cols = colunas_view(view)
    sel = ", ".join(("toString(id_linha) AS id_linha" if c == "id_linha" else q(c)) for c in cols)
    df = consultar(f"SELECT {sel} FROM {view}" + (f" WHERE {onde}" if onde else ""))
    for c in df.columns:
        if c != "id_linha": df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

def executar_robusto(sql, antes=None, tentativas=5, espera=120):
    # Escritas não são repetidas pelo transporte: se faltar memória no servidor (Code 241) ou a conexão cair, esperamos e refazemos (tudo aqui é idempotente).
    for k in range(1, tentativas + 1):
        try:
            if antes: executar(antes)
            return executar(sql)
        except PermissionError:
            raise
        except Exception as e:
            if isinstance(e, ErroConexao) or "241" in str(e) or "memory limit" in str(e).lower():
                print(f"  ⚠ falha por memória/conexão (tentativa {k}/{tentativas}): {str(e)[:160]}; aguardando {espera}s e refazendo...")
                time.sleep(espera); continue
            raise
    raise RuntimeError(f"não consegui concluir após {tentativas} tentativas: {sql[:100]}...")

def distribuir(total, tamanhos, alpha, tetos):
    # Reparte `total` linhas entre os tipos (alpha = 0: partes iguais), sem passar do teto de cada um; o que sobra de um tipo pequeno vai para os demais.
    q_ = {t: 0 for t in tamanhos}; livres = [t for t in tamanhos if tetos[t] > 0]; restante = int(total)
    if sum(tetos[t] for t in livres) < restante: raise ValueError(f"não há linhas suficientes: pedido {restante:,}, disponível {sum(tetos[t] for t in livres):,}")
    while restante > 0 and livres:
        w = {t: float(tamanhos[t]) ** alpha for t in livres}; sw = sum(w.values())
        alvo = {t: restante * w[t] / sw for t in livres}
        cheios = [t for t in livres if q_[t] + alvo[t] >= tetos[t]]
        if not cheios:
            base = {t: int(alvo[t]) for t in livres}; sobra = restante - sum(base.values())
            for t in sorted(livres, key=lambda t: alvo[t] - base[t], reverse=True)[:sobra]: base[t] += 1
            for t in livres: q_[t] += base[t]
            restante = 0
        else:
            for t in cheios:
                restante -= tetos[t] - q_[t]; q_[t] = tetos[t]; livres.remove(t)
    return q_

def _limpar(o):
    if isinstance(o, dict): return {str(k): _limpar(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [_limpar(v) for v in o]
    if isinstance(o, np.ndarray): return _limpar(o.tolist())
    if isinstance(o, (np.integer,)): return int(o)
    if isinstance(o, (np.floating, float)): return None if (o != o) else float(o)
    if isinstance(o, Path): return str(o)
    return o

def salvar_json(obj, caminho):
    Path(caminho).write_text(json.dumps(_limpar(obj), indent=1, ensure_ascii=False), encoding="utf-8")

### Ferramentas do SVM
`_treinar_e_prever` treina o SVM multiclasse (21 pares) e mede treino e previsão; `rodar_tarefas` é o **escalonador de treinos paralelos condicionado à memória**; `executar_com_limite` roda um treino num processo à parte com limite de tempo.

In [5]:
# ---------- ferramentas de métricas, pré-processamento e custo ----------
class Medidor:
    # Mede tempo de relógio, tempo de CPU (todos os núcleos do processo) e pico de RAM (RSS, amostrado a cada 50 ms) de um bloco `with`.
    def __init__(self, intervalo=0.05): self.intervalo = intervalo
    def __enter__(self):
        self.t0, self.c0 = time.perf_counter(), time.process_time(); self._ativo = psutil is not None
        if self._ativo:
            self._p = psutil.Process(); self.base = self._p.memory_info().rss; self.pico = self.base; self._parar = threading.Event()
            def amostrar():
                while not self._parar.is_set():
                    self.pico = max(self.pico, self._p.memory_info().rss); self._parar.wait(self.intervalo)
            self._th = threading.Thread(target=amostrar, daemon=True); self._th.start()
        return self
    def __exit__(self, *a):
        self.relogio, self.cpu = time.perf_counter() - self.t0, time.process_time() - self.c0
        if self._ativo:
            self._parar.set(); self._th.join(); self.pico = max(self.pico, self._p.memory_info().rss)
            self.pico_mib, self.delta_mib = self.pico / 2 ** 20, (self.pico - self.base) / 2 ** 20
        else:
            self.pico_mib = self.delta_mib = float("nan")

class PreprocessadorRF:
    # O pré-processamento do RF, igual ao da view definitiva: ±inf e vazios viram "ausente"; nas colunas com indicador, o ausente é trocado pela mediana do
    # TREINO e entra uma coluna <coluna>_ausente (0/1) no fim. A ordem das colunas do modelo é: atributos, depois os indicadores.
    def __init__(self, features, medianas, retiradas=(), versao_views=""):
        self.features, self.medianas, self.retiradas, self.versao_views = list(features), dict(medianas), list(retiradas), versao_views
        self.indicadores = [c for c in self.features if c in self.medianas]
        self.colunas_modelo = self.features + [c + "_ausente" for c in self.indicadores]
        idx = {c: i for i, c in enumerate(self.features)}
        self._pos = np.array([idx[c] for c in self.indicadores], dtype=int); self._med = np.array([self.medianas[c] for c in self.indicadores], dtype=float)
    def transformar(self, X):
        X = np.asarray(X, dtype=np.float64); X = np.where(np.isinf(X), np.nan, X)
        if len(self._pos) == 0: return X
        sub = X[:, self._pos]; aus = np.isnan(sub); X = X.copy(); X[:, self._pos] = np.where(aus, self._med, sub)
        return np.hstack([X, aus.astype(np.float64)])
    def para_dict(self):
        return {"features": self.features, "indicadores": self.indicadores, "colunas_modelo": self.colunas_modelo, "medianas_treino": self.medianas,
                "colunas_retiradas_ablacao": self.retiradas, "versao_views_rf": self.versao_views,
                "regra": "±inf e vazios = ausente; nas colunas com indicador: ausente -> mediana do treino e coluna <coluna>_ausente (0/1) no fim"}

def medir_inferencia(modelo, prep, X_modelo, X_raw, n_lat=1000, semente=42, repeticoes_lote=3):
    # Latência (1 fluxo por vez) e vazão (lote inteiro), com n_jobs = 1 e com o n_jobs original; com e sem o pré-processamento.
    rng = np.random.default_rng(semente); idx = rng.choice(len(X_modelo), size=min(n_lat, len(X_modelo)), replace=False)
    original = modelo.n_jobs; linhas = []
    for nj in dict.fromkeys([1, original]):
        modelo.set_params(n_jobs=nj)
        for i in idx[:20]: modelo.predict_proba(X_modelo[i:i + 1])                         # aquecimento
        t_mod, t_fim = [], []
        for i in idx:
            r = X_raw[i:i + 1]
            t0 = time.perf_counter(); m = prep.transformar(r); t1 = time.perf_counter(); modelo.predict_proba(m); t2 = time.perf_counter()
            t_mod.append(t2 - t1); t_fim.append(t2 - t0)
        l_mod, l_fim = np.percentile(t_mod, [50, 95, 99]) * 1000, np.percentile(t_fim, [50, 95, 99]) * 1000
        v_mod, v_fim = [], []
        for _ in range(repeticoes_lote):
            t0 = time.perf_counter(); modelo.predict_proba(X_modelo); v_mod.append(len(X_modelo) / (time.perf_counter() - t0))
            t0 = time.perf_counter(); modelo.predict_proba(prep.transformar(X_raw)); v_fim.append(len(X_raw) / (time.perf_counter() - t0))
        linhas.append({"n_jobs": nj, "n_latencia": len(idx), "latencia_modelo_p50_ms": l_mod[0], "latencia_modelo_p95_ms": l_mod[1], "latencia_modelo_p99_ms": l_mod[2],
                       "latencia_fim_a_fim_p50_ms": l_fim[0], "latencia_fim_a_fim_p95_ms": l_fim[1], "latencia_fim_a_fim_p99_ms": l_fim[2],
                       "vazao_modelo_fluxos_s": float(np.median(v_mod)), "vazao_fim_a_fim_fluxos_s": float(np.median(v_fim)), "n_lote": len(X_modelo)})
    modelo.set_params(n_jobs=original)
    return linhas

def tamanho_e_nos(modelo, caminho, compressao):
    joblib.dump(modelo, caminho, compress=compressao)
    return {"arquivo_mib": os.path.getsize(caminho) / 2 ** 20, "n_arvores": len(modelo.estimators_), "nos_total": int(sum(e.tree_.node_count for e in modelo.estimators_)),
            "profundidade_media": float(np.mean([e.get_depth() for e in modelo.estimators_])), "profundidade_max": int(max(e.get_depth() for e in modelo.estimators_))}

# ---------- métricas multiclasse (a partir da matriz de confusão; aceitam pesos por linha) ----------
def metricas_multi(y, proba, w=None):
    y = np.asarray(y, dtype=int); proba = np.asarray(proba, dtype=float); K_ = proba.shape[1]; pred = proba.argmax(axis=1); w_ = None if w is None else np.asarray(w, dtype=float)
    cm = np.zeros((K_, K_)); np.add.at(cm, (y, pred), 1.0 if w_ is None else w_)
    s, t, p, d = cm.sum(), cm.sum(axis=1), cm.sum(axis=0), np.diag(cm)                  # total, linhas por classe verdadeira, previstas por classe, acertos
    rec = np.divide(d, t, out=np.full(K_, np.nan), where=t > 0); prec = np.divide(d, p, out=np.zeros(K_), where=p > 0); f1 = np.divide(2 * d, t + p, out=np.zeros(K_), where=(t + p) > 0)
    den = math.sqrt(max((s * s - (p * p).sum()) * (s * s - (t * t).sum()), 0.0)); mcc = float((d.sum() * s - (p * t).sum()) / den) if den > 0 else 0.0
    auc_k = np.full(K_, np.nan)
    for k in range(K_):
        yk = (y == k).astype(int)
        if 0 < yk.sum() < len(yk): auc_k[k] = roc_auc_score(yk, proba[:, k], sample_weight=w_)
    esc = {"acuracia": float(d.sum() / s), "acuracia_balanceada": float(np.nanmean(rec)), "precisao_macro": float(prec.mean()), "f1_macro": float(f1.mean()), "f1_ponderado": float((f1 * t).sum() / t.sum()), "mcc": mcc, "roc_auc_ovr_macro": float(np.nanmean(auc_k))}
    return esc, {"cm": cm, "recall": rec, "precisao": prec, "f1": f1, "auc": auc_k, "suporte": t}

# autoteste das ferramentas (não toca no servidor): confere as métricas multiclasse contra o scikit-learn
from sklearn.metrics import accuracy_score as _acc, balanced_accuracy_score as _bacc, f1_score as _f1, matthews_corrcoef as _mcc, precision_score as _prec
_r = np.random.default_rng(0); _y = _r.integers(0, 4, 400); _P = _r.dirichlet(np.ones(4), 400); _P[np.arange(400), _y] += 0.35; _P /= _P.sum(axis=1, keepdims=True); _w = _r.uniform(0.5, 3, 400); _pr = _P.argmax(axis=1)
for _ww in (None, _w):
    _e, _a = metricas_multi(_y, _P, _ww)
    assert abs(_e["acuracia"] - _acc(_y, _pr, sample_weight=_ww)) < 1e-12 and abs(_e["acuracia_balanceada"] - _bacc(_y, _pr, sample_weight=_ww)) < 1e-12, _e
    assert abs(_e["f1_macro"] - _f1(_y, _pr, average="macro", sample_weight=_ww)) < 1e-12 and abs(_e["f1_ponderado"] - _f1(_y, _pr, average="weighted", sample_weight=_ww)) < 1e-12, _e
    assert abs(_e["mcc"] - _mcc(_y, _pr, sample_weight=_ww)) < 1e-12 and abs(_e["precisao_macro"] - _prec(_y, _pr, average="macro", sample_weight=_ww, zero_division=0)) < 1e-12, _e
    assert abs(_e["roc_auc_ovr_macro"] - roc_auc_score(_y, _P, multi_class="ovr", average="macro", sample_weight=_ww)) < 1e-12, _e
assert abs(metricas_multi(np.array([0, 1, 2]), np.eye(3))[0]["mcc"] - 1.0) < 1e-12
print("Autoteste das ferramentas OK | sklearn", sklearn.__version__, "| psutil", "ausente" if psutil is None else psutil.__version__)

import gc, threading
def rss_mib():
    return psutil.Process().memory_info().rss / 2 ** 20 if psutil else float("nan")

def fmt_tempo(s):
    if s != s or s == float("inf"): return "?"
    s = int(round(s)); h, r = divmod(s, 3600); m, sg = divmod(r, 60)
    return f"{h}h {m:02d}min" if h else (f"{m}min {sg:02d}s" if m else f"{sg}s")

class Barra:
    # Barra de progresso: HTML no Jupyter/VS Code; se não houver IPython, imprime uma linha a cada 5% ou 60 s.
    def __init__(self, total, desc="", intervalo=0.4):
        self.total, self.desc, self.n, self.t0, self.intervalo = max(int(total), 1), desc, 0, time.perf_counter(), intervalo
        self._ult, self._ult_pct, self._extra, self._h = 0.0, -1, "", None
        try:
            from IPython.display import display as _d, HTML as _H
            self._H = _H; self._h = _d(_H(self._html()), display_id=True)
        except Exception:
            self._h = None
    def _info(self):
        dec = time.perf_counter() - self.t0; frac = min(self.n / self.total, 1.0)
        return frac, dec, (dec / self.n * (self.total - self.n) if self.n else float("nan"))
    def _linha(self):
        frac, dec, eta = self._info()
        return f"{self.desc}: {100 * frac:5.1f}% ({self.n}/{self.total}) | decorrido {fmt_tempo(dec)} | restante ~{fmt_tempo(eta)}" + (f" | {self._extra}" if self._extra else "")
    def _html(self):
        frac, dec, eta = self._info()
        return (f"<div style='font-family:monospace;font-size:13px'><b>{self.desc}</b> <progress value='{self.n}' max='{self.total}' style='width:260px;vertical-align:middle'></progress> "
                f"<b>{100 * frac:5.1f}%</b> ({self.n}/{self.total}) &nbsp;|&nbsp; decorrido {fmt_tempo(dec)} &nbsp;|&nbsp; restante ~{fmt_tempo(eta)}"
                + (f"<br><span style='color:#555'>{self._extra}</span>" if self._extra else "") + "</div>")
    def _mostrar(self, forcar=False):
        agora = time.perf_counter()
        if self._h is not None:
            if forcar or agora - self._ult >= self.intervalo: self._h.update(self._H(self._html())); self._ult = agora
        else:
            pct = int(100 * self.n / self.total) // 5
            if forcar or pct != self._ult_pct or agora - self._ult >= 60: print(self._linha(), flush=True); self._ult_pct, self._ult = pct, agora
    def avancar(self, n=1, extra=None):
        self.n += n
        if extra is not None: self._extra = extra
        self._mostrar(forcar=self.n >= self.total)
    def fechar(self, extra=""):
        self._extra = extra or self._extra; self._mostrar(forcar=True)
        if self._h is not None: print(f"{self.desc}: concluído em {fmt_tempo(time.perf_counter() - self.t0)}")

# ---------- ferramentas do SVM multiclasse ----------
def met(y, P, w=None):
    # Métricas multiclasse do notebook 05m (aqui P são os ESCORES do SVM, não probabilidades: a classe é o maior escore e o ROC-AUC é um-contra-resto por classe, que só usa a ordem dos escores).
    e, a = metricas_multi(y, P, w); e = dict(e); e["roc_auc"] = e.pop("roc_auc_ovr_macro"); return e, a

def recall_por_classe(y, P):
    y = np.asarray(y, dtype=int); K_ = P.shape[1]; pred = np.asarray(P).argmax(axis=1); cm_ = np.bincount(y * K_ + pred, minlength=K_ * K_).reshape(K_, K_).astype(float)
    return np.divide(np.diag(cm_), cm_.sum(axis=1), out=np.full(K_, np.nan), where=cm_.sum(axis=1) > 0)

def novo_pipeline(C, gamma, cache=None):
    # SVM multiclasse completo: padroniza SÓ as colunas numéricas (média e desvio do próprio treino; os indicadores _ausente ficam 0/1) e treina o SVM RBF (21 pares), com class_weight="balanced" como no RF.
    pre = ColumnTransformer([("num", StandardScaler(), NUM)] + ([("ind", "passthrough", IND)] if IND else []))
    return Pipeline([("escala", pre), ("svm", SVC(kernel="rbf", C=float(C), gamma=float(gamma), cache_size=cache or CACHE_MB, class_weight="balanced"))])

def _treinar_e_prever(Xtr, ytr, Xte, C, gamma, caminho_modelo, cache):
    pipe = novo_pipeline(C, gamma, cache)
    with Medidor() as m: pipe.fit(Xtr, ytr)
    svm, pre = pipe.named_steps["svm"], pipe.named_steps["escala"]
    t0 = time.perf_counter(); Xt = pre.transform(Xte); t_tr = time.perf_counter() - t0
    t0 = time.perf_counter(); sc = svm.decision_function(Xt); t_sv = time.perf_counter() - t0                  # escores um-contra-resto (n x K); a classe é o argmax
    nsv = int(svm.n_support_.sum())
    meta = {"n_treino": int(len(ytr)), "n_avaliacao": int(len(Xte)), "C": float(C), "gamma": float(gamma), "tempo_treino_s": m.relogio, "cpu_s": m.cpu, "pico_ram_mib": m.pico_mib, "acrescimo_ram_mib": m.delta_mib,
            "n_sv": nsv, "n_sv_por_classe": [int(v) for v in svm.n_support_], "tempo_transformacao_s": t_tr, "tempo_previsao_s": t_sv, "ns_por_par": t_sv / (len(Xte) * max(nsv, 1)) * 1e9}
    if caminho_modelo:                                                                                          # o modelo é gravado DENTRO do processo (não volta pelo canal de comunicação) e antes do marcador do treino
        pipe.colunas_modelo_ = list(CM); tmp = Path(str(caminho_modelo) + ".tmp"); joblib.dump(pipe, tmp, compress=COMPRESSAO_MODELO); os.replace(tmp, caminho_modelo)
    return meta, sc.astype(np.float32)

def _worker_dobra(dir_dados, itr, ite, C, gamma, caminho_modelo=None, cache=None):
    # Lê os dados do disco (memória mapeada: o processo principal não copia nada) e treina. itr/ite = None: treino com o ajuste inteiro e avaliação na validação.
    Xa = np.load(Path(dir_dados) / "X_aj.npy", mmap_mode="r"); ya = np.load(Path(dir_dados) / "y_aj.npy")
    Xtr, ytr = (np.ascontiguousarray(Xa[itr]), ya[itr]) if itr is not None else (np.ascontiguousarray(Xa), ya)
    Xte = np.ascontiguousarray(Xa[ite]) if ite is not None else np.load(Path(dir_dados) / "X_va.npy")
    return _treinar_e_prever(Xtr, ytr, Xte, C, gamma, caminho_modelo, cache)

def _worker_arrays(Xtr, ytr, Xte, C, gamma, caminho_modelo=None, cache=None): return _treinar_e_prever(Xtr, ytr, Xte, C, gamma, caminho_modelo, cache)

def ram_livre_gib():
    s = globals().get("RAM_LIVRE_SIMULADA")                                                                     # só para testes
    return float(s() if callable(s) else s) if s is not None else (psutil.virtual_memory().available / 2 ** 30 if psutil else float("inf"))

def processos_seguros(desejado, ram_por_proc_gib):
    # Quantos processos simultâneos cabem AGORA: cada um precisa de ram_por_proc × 1,3, e a reserva nunca é tocada. Sempre devolve pelo menos 1 (nunca trava), mas avisa se nem 1 cabe folgado.
    livre = ram_livre_gib() - RAM_RESERVA_GIB
    if not np.isfinite(livre): return int(desejado)
    return int(max(1, min(desejado, livre // (ram_por_proc_gib * 1.3))))

def rodar_tarefas(pendentes, lancar, coletar, barra, limite_s, k_max, ram_proc, rotulo, parar_apos=None):
    # Escalonador de treinos independentes, UM PROCESSO NOVO por treino. Só inicia mais um treino se houver memória livre para ele (ram_proc × 1,3 + reserva) e se o anterior já ocupou a sua memória;
    # com um treino só rodando, sempre inicia o próximo (nunca trava). Se a memória ficar crítica com vários treinos, o mais novo é interrompido e volta para a fila. Treino acima do limite de tempo é encerrado.
    espera_desde = None; fila, rodando, feitos, ultima, stats = collections.deque(pendentes), [], 0, 0.0, {"max_simultaneos": 0, "interrompidos_por_memoria": 0, "ram_proc_final": ram_proc}
    try:
        while fila or rodando:
            while fila and len(rodando) < k_max:
                livre = ram_livre_gib(); agora = time.perf_counter()
                if rodando and (agora - ultima < INTERVALO_PARTIDAS_S or livre - RAM_RESERVA_GIB < ram_proc * 1.3): break
                if not rodando:
                    if livre < RAM_MINIMA_UM_TREINO_GIB:                                  # com nada rodando, espera a memória liberar (até 10 min); depois, para com uma mensagem clara
                        espera_desde = espera_desde or agora
                        if agora - espera_desde > 600: raise RuntimeError(f"Falta memória livre: um treino precisa de pelo menos {RAM_MINIMA_UM_TREINO_GIB:.1f} GiB livres e há {livre:.1f} GiB. Feche o Meet, o Edge e os outros notebooks (ou defina RAM_MINIMA_UM_TREINO_GIB mais baixo, por sua conta e risco).")
                        barra._extra = f"aguardando memória livre ({livre:.1f} GiB; preciso de pelo menos {RAM_MINIMA_UM_TREINO_GIB:.1f} GiB)..."; barra._mostrar(); time.sleep(2.0); break
                    espera_desde = None
                t = fila.popleft(); ex, fut = lancar(t); rodando.append({"t": t, "ex": ex, "fut": fut, "t0": time.perf_counter(), "k": len(rodando) + 1}); ultima = time.perf_counter(); stats["max_simultaneos"] = max(stats["max_simultaneos"], len(rodando))
            for r in list(rodando):
                dec = time.perf_counter() - r["t0"]
                if r["fut"].done():
                    res = r["fut"].result(); r["ex"].shutdown(wait=True); rodando.remove(r); feitos += 1; ram_proc = max(ram_proc, 1.15 * res[0]["pico_ram_mib"] / 1024); stats["ram_proc_final"] = ram_proc; coletar(r["t"], res, r["k"])
                    if parar_apos and feitos >= parar_apos: raise RuntimeError("parada de teste (PARAR_APOS_FITS)")
                elif dec > limite_s: r["ex"].shutdown(wait=True, kill_workers=True); rodando.remove(r); feitos += 1; coletar(r["t"], None, r["k"])
            if len(rodando) > 1 and ram_livre_gib() < RAM_CRITICA_GIB:
                r = rodando.pop(); r["ex"].shutdown(wait=True, kill_workers=True); fila.appendleft(r["t"]); k_max = max(1, k_max - 1); stats["interrompidos_por_memoria"] += 1
                print(f"  ⚠ memória livre crítica ({ram_livre_gib():.2f} GiB): um treino foi interrompido e volta para a fila; o limite de treinos simultâneos agora é {k_max}")
            barra._extra = f"{rotulo} | {len(rodando)} treino(s) rodando (até {k_max}) | memória livre {ram_livre_gib():.1f} GiB | " + (f"mais antigo há {fmt_tempo(max(time.perf_counter() - r['t0'] for r in rodando))}" if rodando else "")
            barra._mostrar(); time.sleep(0.5)
    except BaseException:
        for r in rodando: r["ex"].shutdown(wait=True, kill_workers=True)
        raise
    return stats



def executar_com_limite(fn, args, limite_s, barra=None, rotulo=""):
    # Um processo NOVO e independente por chamada; se passar do limite, é encerrado (devolve None). Espera em fatias de 0,5 s: o botão "interromper" funciona e a barra mostra o tempo decorrido.
    ex = LokyPool(max_workers=1); fut = ex.submit(fn, *args); t0 = time.perf_counter()
    try:
        while True:
            decorrido = time.perf_counter() - t0
            if decorrido >= limite_s: ex.shutdown(wait=True, kill_workers=True); return None
            if barra is not None: barra._extra = f"{rotulo} | em andamento há {fmt_tempo(decorrido)} (limite {fmt_tempo(limite_s)})"; barra._mostrar()
            try: res = fut.result(timeout=min(0.5, limite_s - decorrido))
            except (concurrent.futures.TimeoutError, TimeoutError): continue
            ex.shutdown(wait=True); return res
    except BaseException:
        ex.shutdown(wait=True, kill_workers=True); raise

def tamanho_do_arquivo(modelo, caminho):
    # Tamanho do arquivo do modelo e dos vetores de suporte, SEM regravar o arquivo (ele já existe).
    svm = modelo.named_steps["svm"]
    return {"arquivo_mib": os.path.getsize(caminho) / 2 ** 20, "n_vetores_suporte": int(svm.n_support_.sum()), "n_sv_por_classe": [int(v) for v in svm.n_support_], "n_atributos": int(svm.shape_fit_[1]), "C": float(svm.C), "gamma": float(svm._gamma), "kernel": svm.kernel}

def _dec(modelo, X): return modelo.decision_function(X)
def prever_paralelo(modelo, X, k):
    # Escores (distância à fronteira) em k processos: o lote é dividido em k pedaços (o resultado é idêntico ao sequencial).
    if k <= 1: return modelo.decision_function(X)
    return np.concatenate(Parallel(n_jobs=k)(delayed(_dec)(modelo, c) for c in np.array_split(X, k)))

def medir_inferencia_svm(modelo, prep, X_modelo, X_raw, n_lat=1000, semente=42, n_lote=20_000, repeticoes_lote=2, k_par=2):
    # Latência (1 fluxo por vez, 1 processo) e vazão (lote) com 1 e com k_par processos; com e sem o pré-processamento (mediana do treino + indicadores). O modelo inclui a padronização.
    rng = np.random.default_rng(semente); idx = rng.choice(len(X_modelo), size=min(n_lat, len(X_modelo)), replace=False)
    for i in idx[:20]: modelo.decision_function(X_modelo[i:i + 1])                           # aquecimento
    t_mod, t_fim = [], []
    for i in idx:
        r = X_raw[i:i + 1]; t0 = time.perf_counter(); m = prep.transformar(r); t1 = time.perf_counter(); modelo.decision_function(m); t2 = time.perf_counter(); t_mod.append(t2 - t1); t_fim.append(t2 - t0)
    l_mod, l_fim = np.percentile(t_mod, [50, 95, 99]) * 1000, np.percentile(t_fim, [50, 95, 99]) * 1000
    Xl, Xr = X_modelo[:n_lote], X_raw[:n_lote]; linhas = []
    for k in dict.fromkeys([1, max(int(k_par), 1)]):
        if k > 1: Parallel(n_jobs=k)(delayed(time.sleep)(0.01) for _ in range(k))
        v_mod, v_fim = [], []
        for _ in range(repeticoes_lote):
            t0 = time.perf_counter(); prever_paralelo(modelo, Xl, k); v_mod.append(len(Xl) / (time.perf_counter() - t0))
            t0 = time.perf_counter(); prever_paralelo(modelo, prep.transformar(Xr), k); v_fim.append(len(Xr) / (time.perf_counter() - t0))
        nan = float("nan")
        linhas.append({"n_jobs": k, "n_latencia": len(idx), "latencia_modelo_p50_ms": l_mod[0] if k == 1 else nan, "latencia_modelo_p95_ms": l_mod[1] if k == 1 else nan, "latencia_modelo_p99_ms": l_mod[2] if k == 1 else nan,
                       "latencia_fim_a_fim_p50_ms": l_fim[0] if k == 1 else nan, "latencia_fim_a_fim_p95_ms": l_fim[1] if k == 1 else nan, "latencia_fim_a_fim_p99_ms": l_fim[2] if k == 1 else nan,
                       "vazao_modelo_fluxos_s": float(np.median(v_mod)), "vazao_fim_a_fim_fluxos_s": float(np.median(v_fim)), "n_lote": len(Xl)})
    return linhas
print("Ferramentas do SVM multiclasse prontas")


Autoteste das ferramentas OK | sklearn 1.9.1 | psutil 7.2.2
Ferramentas do SVM multiclasse prontas


## 1. Parâmetros e o que existe
Tudo o que define a busca fica aqui: a grade, as dobras, a regra de escolha e os limites do paralelismo. **Não mude nada depois de ver resultados.**

In [6]:
# ---------- 1.1 Parâmetros ----------
import hashlib, io, copy, collections, shutil
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
from joblib import Parallel, delayed
from joblib.externals.loky import ProcessPoolExecutor as LokyPool
import concurrent.futures

GRADE = globals().get("GRADE", {"C": (1.0, 10.0, 100.0), "gamma_fator": (0.25, 1.0, 4.0)})    # a grade do binário (gamma = múltiplos do valor "scale")
K_CV = globals().get("K_CV", 5)
SEMENTE = globals().get("SEMENTE", 42)                                 # dobras da validação cruzada, partições da validação e amostras de latência (a mesma do RF)
K_PARTICOES_VAL = globals().get("K_PARTICOES_VAL", 5)
REPETICOES_TEMPO = globals().get("REPETICOES_TEMPO", 1)                # refits cronometrados (o RF usou 3; cada refit do SVM leva ~1 h, então o padrão aqui é 1)
N_LATENCIA = globals().get("N_LATENCIA", 1000)
COMPRESSAO_MODELO = globals().get("COMPRESSAO_MODELO", 3)
CACHE_MB = globals().get("CACHE_MB", 200)                              # cache do kernel de cada SVM (por processo; medido nos notebooks 10 e 10b)
LIMITE_FIT_S = globals().get("LIMITE_FIT_S", 4 * 3600)                 # nenhum treino pode passar disto (o processo é encerrado e a configuração fica marcada)
# --- paralelismo condicionado à memória ---
PROCESSOS_MAX = globals().get("PROCESSOS_MAX", 4)                      # teto de treinos simultâneos: SÓ são usados se houver memória livre para cada um
RAM_POR_PROCESSO_GIB = globals().get("RAM_POR_PROCESSO_GIB", 1.0)      # estimativa inicial de RAM de cada treino (a calibração mediu ~0,5 GiB com 111 mil linhas); é atualizada com o pico medido
RAM_RESERVA_GIB = globals().get("RAM_RESERVA_GIB", 1.0)                # memória livre que NUNCA é tocada (sistema e outros programas)
RAM_MINIMA_UM_TREINO_GIB = globals().get("RAM_MINIMA_UM_TREINO_GIB", 0.7)        # memória livre mínima para rodar UM treino (com nada rodando, espera até 10 min por ela e depois para com uma mensagem)
RAM_CRITICA_GIB = globals().get("RAM_CRITICA_GIB", 0.4)                # se a memória livre cair abaixo disto com vários treinos rodando, o mais novo é interrompido e volta para a fila
INTERVALO_PARTIDAS_S = globals().get("INTERVALO_PARTIDAS_S", 10)       # espera entre a partida de um treino e a decisão de iniciar outro (a memória do primeiro leva alguns segundos para subir)
PROCESSOS_PREVISAO = globals().get("PROCESSOS_PREVISAO", 2)            # processos para prever em lote (ganho de 1,84× medido no notebook 10), também limitado pela memória
N_LOTE_VAZAO = globals().get("N_LOTE_VAZAO", 5_000)                    # linhas usadas para medir a vazão em lote (o SVM multiclasse é lento: ~1 min por medida com 5 mil linhas)
RAM_PREVISAO_POR_PROCESSO_GIB = globals().get("RAM_PREVISAO_POR_PROCESSO_GIB", 0.5)     # RAM de cada processo de previsão (o modelo e um pedaço do lote)
REPETICOES_LOTE = globals().get("REPETICOES_LOTE", 2)
AVALIAR_VALIDACAO_POR_CONFIG = globals().get("AVALIAR_VALIDACAO_POR_CONFIG", True)   # treino completo (ajuste inteiro) de cada configuração, avaliado na validação (como no RF)
COMPARAR_COM_PADRAO = globals().get("COMPARAR_COM_PADRAO", True)       # refit também do padrão (C = 1, gamma = 1×), para a comparação no teste
MULT_ERRO_PADRAO = globals().get("MULT_ERRO_PADRAO", 1.0)
METRICAS_DECISAO = ("roc_auc", "mcc")                                   # ROC-AUC um-contra-resto (macro) e MCC multiclasse: as mesmas do RF
PADRAO = {"C": 1.0, "gamma_fator": 1.0}                                 # o padrão do scikit-learn (C = 1, gamma = scale)
PARAR_APOS_FITS = globals().get("PARAR_APOS_FITS", None)                # só para testes
NOVA_EXECUCAO = globals().get("NOVA_EXECUCAO", False)                   # True = não retomar uma busca interrompida
assert set(GRADE) == {"C", "gamma_fator"} and all(len(v) >= 2 for v in GRADE.values()), "GRADE precisa ter C e gamma_fator, cada um com 2 valores ou mais"

nome_cfg = lambda C, f: f"C{C:g}_g{f:g}"; rot_cfg = lambda C, f: f"C={C:g}|gamma={f:g}x"
CONFIGS = sorted([(float(C), float(f)) for C in GRADE["C"] for f in GRADE["gamma_fator"]], key=lambda c: (c != (PADRAO["C"], PADRAO["gamma_fator"]), c))     # o padrão primeiro
assert (PADRAO["C"], PADRAO["gamma_fator"]) in CONFIGS or not COMPARAR_COM_PADRAO, "o padrão (C = 1, gamma = 1×) precisa estar na grade para a comparação"
N_FITS = len(CONFIGS) * (K_CV + (1 if AVALIAR_VALIDACAO_POR_CONFIG else 0))
ARTEFATOS_BASE = Path(globals().get("REPO_ROOT", Path.cwd())) / "artefatos_modelos" / "svm_multiclasse_busca"; ARTEFATOS_BASE.mkdir(parents=True, exist_ok=True)

V_AJUSTE, V_VALID, V_REFIT = "prep_vw_bal_multi_treino", "prep_vw_bal_multi_validacao", "prep_vw_bal_multi_refit"
SEL_TAB, DADOS_TAB, TAB_REFIT = "prep_bal_multi_selecao", "prep_bal_multi_dados", "prep_bal_multi_refit_sel"
AUD_TAB, TAB_CLASSES = "prep_views_rf_multi", "prep_classes_rf_multi"
faltam = [t for t in ("prep_final_treino", V_AJUSTE, V_VALID, V_REFIT, SEL_TAB, DADOS_TAB, AUD_TAB, TAB_CLASSES, TAB_REFIT) if not existe(t)]
assert not faltam, f"Rode antes os notebooks 02, 03m, 04m e 05m: faltam {faltam}"
versao_sel = str(consultar(f"SELECT any(versao) AS v FROM {SEL_TAB}")["v"][0])
aud = consultar_num(f"SELECT posicao, feature, situacao, mediana_imputacao, tem_indicador, versao FROM {AUD_TAB} ORDER BY posicao"); versao_views = str(aud["versao"].iloc[0])
cl = consultar_num(f"SELECT codigo, classe, linhas_treino FROM {TAB_CLASSES} ORDER BY codigo")
CLASSES = cl.classe.tolist(); K = len(CLASSES); COD = {c: int(k) for c, k in zip(cl.classe, cl.codigo)}; N_TREINO = {c: int(n) for c, n in zip(cl.classe, cl.linhas_treino)}
assert [COD[c] for c in CLASSES] == list(range(K)), "os códigos das classes precisam ser 0..K-1"
cont = consultar_num(f"SELECT papel, y, count() AS n FROM {SEL_TAB} GROUP BY papel, y"); cont = {(r.papel, int(r.y)): int(r.n) for r in cont.itertuples()}
N_AJ_C = {c: cont[("treino", COD[c])] for c in CLASSES}; N_VA_C = {c: cont[("validacao", COD[c])] for c in CLASSES}
print(f"Seleção do notebook 04m (versão {versao_sel}) | views do notebook 03m (versão {versao_views})")
print(f"{K} classes: {CLASSES}"); print(pd.DataFrame({"no treino": N_TREINO, "ajuste": N_AJ_C, "validação": N_VA_C}).T.to_string())
print(f"\nGrade: {len(CONFIGS)} configurações = {len(GRADE['C'])} (C {list(GRADE['C'])}) × {len(GRADE['gamma_fator'])} (gamma {list(GRADE['gamma_fator'])} × o gamma de referência) | validação cruzada: {K_CV} dobras | sem sementes (o SVM é determinístico) | class_weight = balanced")
print(f"Treinos: {len(CONFIGS)} configurações × ({K_CV} dobras" + (" + 1 treino completo avaliado na validação" if AVALIAR_VALIDACAO_POR_CONFIG else "") + f") = {N_FITS} SVMs (cada um com os 21 pares de classes) | limite por treino {LIMITE_FIT_S / 3600:.1f} h | cache {CACHE_MB} MB por processo")
print(f"Paralelismo: até {PROCESSOS_MAX} treinos ao mesmo tempo, SÓ se a memória livre permitir (cada treino ~{RAM_POR_PROCESSO_GIB:.1f} GiB × 1,3, mais {RAM_RESERVA_GIB:.1f} GiB de reserva)")
T_SEC = {}


Seleção do notebook 04m (versão 20261027_173009) | views do notebook 03m (versão 20261026_010742)
7 classes: ['BENIGN', 'LDAP', 'MSSQL', 'NetBIOS', 'Syn', 'UDP', 'UDPLag']
           BENIGN   LDAP   MSSQL  NetBIOS     Syn      UDP  UDPLag
no treino   50768  30099  207003    19490  155515  1075337   89066
ajuste      40000  24079   40000    15592   40000    40000   40000
validação   10000   6020   10000     3898   10000    10000   10000

Grade: 9 configurações = 3 (C [1.0, 10.0, 100.0]) × 3 (gamma [0.25, 1.0, 4.0] × o gamma de referência) | validação cruzada: 5 dobras | sem sementes (o SVM é determinístico) | class_weight = balanced
Treinos: 9 configurações × (5 dobras + 1 treino completo avaliado na validação) = 54 SVMs (cada um com os 21 pares de classes) | limite por treino 4.0 h | cache 200 MB por processo
Paralelismo: até 4 treinos ao mesmo tempo, SÓ se a memória livre permitir (cada treino ~1.0 GiB × 1,3, mais 1.0 GiB de reserva)


## 2. Dados, pré-processamento, classes e dobras
Lê o ajuste, a validação e o refit (as mesmas views do RF), ordena por `id_linha`, confere o pré-processamento e **confere que as dobras são as do RF** (impressão digital). Calcula o gamma de referência (`scale`) **uma vez**, no ajuste inteiro. A busca guarda tudo numa pasta cuja impressão digital depende da grade, das dobras e dos dados: se for interrompida, rodar de novo **retoma**.

In [7]:
# ---------- 2.1 Dados, pré-processamento, classes, dobras e gamma de referência ----------
USADAS, RETIRADAS = aud[aud.situacao == "usada"], aud[aud.situacao != "usada"].feature.tolist()
MEDIANAS = {r.feature: float(r.mediana_imputacao) for r in USADAS.itertuples() if int(r.tem_indicador) == 1}
prep = PreprocessadorRF(USADAS.feature.tolist(), MEDIANAS, RETIRADAS, versao_views)
assert prep.colunas_modelo == [c for c in colunas_view(V_AJUSTE) if c not in ("id_linha", "y")], "a ordem das colunas da view difere da do pré-processamento"
print(f"Atributos do modelo: {len(prep.colunas_modelo)} ({len(prep.features)} + {len(prep.indicadores)} indicadores) | retiradas na ablação: {RETIRADAS}")

def ordenar(df): return df.iloc[np.argsort(df["id_linha"].astype("uint64").to_numpy(), kind="stable")].reset_index(drop=True)
aj, va, rf_df = ordenar(ler_view(V_AJUSTE)), ordenar(ler_view(V_VALID)), ordenar(ler_view(V_REFIT))             # a MESMA ordem do RF (07m)
CM = prep.colunas_modelo; NUM, IND = list(range(len(prep.features))), list(range(len(prep.features), len(CM)))        # as colunas numéricas vêm primeiro; os indicadores _ausente, no fim
X_aj, y_aj = aj[CM].to_numpy(float), aj["y"].to_numpy(int)
X_va, y_va, w_va = va[CM].to_numpy(float), va["y"].to_numpy(int), va["peso"].to_numpy(float)
X_rf, y_rf = rf_df[CM].to_numpy(float), rf_df["y"].to_numpy(int)
for nome, X in (("ajuste", X_aj), ("validação", X_va), ("refit", X_rf)): assert np.isfinite(X).all(), f"valores não finitos em {nome}"
X_va_raw = X_va[:, :len(prep.features)].copy()
for j, c in enumerate(prep.indicadores): X_va_raw[X_va[:, len(prep.features) + j] == 1, prep.features.index(c)] = np.nan
assert np.allclose(prep.transformar(X_va_raw), X_va, rtol=1e-12, atol=1e-12), "o PreprocessadorRF não reproduz a matriz da view"
conta = lambda y: {CLASSES[k]: int((y == k).sum()) for k in range(K)}
assert conta(y_aj) == N_AJ_C and conta(y_va) == N_VA_C and len(X_rf) == sum(N_AJ_C.values()) + sum(N_VA_C.values()), "as linhas por classe não coincidem com a seleção do notebook 04m"
assert all(abs(float(w_va[y_va == COD[c]].sum()) - N_TREINO[c]) < 1e-6 * N_TREINO[c] for c in CLASSES), "os pesos da validação não somam as linhas de cada classe no treino"

DOBRAS = list(StratifiedKFold(K_CV, shuffle=True, random_state=SEMENTE).split(X_aj, y_aj))                       # IGUAL ao notebook 07m
assert all(len(np.unique(y_aj[ite])) == K for _, ite in DOBRAS), "alguma dobra ficou sem alguma classe"
_a = np.zeros(len(y_aj), np.int8)
for k_, (_, ite_) in enumerate(DOBRAS): _a[ite_] = k_
IMPRESSAO_DOBRAS = format(zlib.crc32(_a.tobytes()), "08x")
tab_classes = pd.DataFrame({"ajuste": np.bincount(y_aj, minlength=K), "validação": np.bincount(y_va, minlength=K)}, index=CLASSES)
print(f"\nAjuste: {len(X_aj):,} linhas | validação: {len(X_va):,} | refit: {len(X_rf):,} | {K} classes"); display(tab_classes.T)
print(f"Dobras: {K_CV}, estratificadas por classe (~{len(X_aj) // K_CV:,} linhas de avaliação cada) | impressão digital das dobras: {IMPRESSAO_DOBRAS}")
RF_IMPRESSAO = None
for p in sorted((ARTEFATOS_BASE.parent / "rf_multiclasse_busca").glob("rfm_busca_*"), reverse=True):                 # as dobras do RF (07m), para conferir que são as mesmas
    try:
        v = json.loads((p / "config_busca.json").read_text(encoding="utf-8")).get("impressao_dobras")
        if v: RF_IMPRESSAO = v; break
    except (OSError, ValueError): pass
if RF_IMPRESSAO: print(f"{'✔' if RF_IMPRESSAO == IMPRESSAO_DOBRAS else '✘'} impressão digital das dobras do RF (07m): {RF_IMPRESSAO} → {'IGUAIS: RF e SVM podem ser comparados dobra a dobra' if RF_IMPRESSAO == IMPRESSAO_DOBRAS else 'DIFERENTES'}")
assert RF_IMPRESSAO is None or RF_IMPRESSAO == IMPRESSAO_DOBRAS, "as dobras deste notebook não são as do RF (07m): confira os dados e a semente"

# gamma de referência: 1 / (colunas × variância dos dados já normalizados), o mesmo cálculo do "scale" do scikit-learn, feito UMA vez no ajuste inteiro
_pre = ColumnTransformer([("num", StandardScaler(), NUM)] + ([("ind", "passthrough", IND)] if IND else [])).fit(X_aj); _Xt = _pre.transform(X_aj)
GAMMA0 = float(1.0 / (_Xt.shape[1] * _Xt.var())); del _Xt, _pre
GAMAS = {f: GAMMA0 * f for f in GRADE["gamma_fator"]}
print(f"gamma de referência (scale) = {GAMMA0:.5f} | gammas da grade: " + ", ".join(f"{f:g}× = {g:.5f}" for f, g in GAMAS.items()) + " (os mesmos valores absolutos valem no refit)")

# pasta da execução: retoma uma busca interrompida com a MESMA configuração e os MESMOS dados
crc = lambda s: zlib.crc32(",".join(s).encode())
IMPRESSAO = hashlib.sha1(json.dumps(_limpar({"grade": GRADE, "k_cv": K_CV, "semente": SEMENTE, "versao_sel": versao_sel, "versao_views": versao_views, "dobras": IMPRESSAO_DOBRAS, "gamma0": round(GAMMA0, 10), "class_weight": "balanced",
                                           "ids_aj": crc(aj["id_linha"]), "ids_va": crc(va["id_linha"]), "avaliar_val": AVALIAR_VALIDACAO_POR_CONFIG}), sort_keys=True).encode()).hexdigest()[:12]
PASTA, finalizadas = None, []
for p in sorted(ARTEFATOS_BASE.glob("svm_busca_*"), reverse=True):
    try: c_ = json.loads((p / "config_busca.json").read_text(encoding="utf-8"))
    except (OSError, ValueError): continue
    if (p / "FINALIZADO").exists(): finalizadas.append((p, c_.get("impressao"))); continue
    if not NOVA_EXECUCAO and PASTA is None and c_.get("impressao") == IMPRESSAO: PASTA = p
if PASTA is None and not NOVA_EXECUCAO and any(imp == IMPRESSAO for _, imp in finalizadas):
    raise RuntimeError("Já existe uma busca FINALIZADA com esta mesma configuração e os mesmos dados: " + ", ".join(p.name for p, imp in finalizadas if imp == IMPRESSAO) + ". Os resultados estão nos arquivos dessa pasta. Para refazer a busca DE PROPÓSITO, defina NOVA_EXECUCAO = True antes desta célula.")
RETOMANDO = PASTA is not None
if PASTA is None:
    RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S"); PASTA = ARTEFATOS_BASE / f"svm_busca_{RUN_ID}"; PASTA.mkdir(parents=True)
    salvar_json({"run_id": RUN_ID, "criado_em": datetime.now().isoformat(timespec="seconds"), "impressao": IMPRESSAO, "grade": GRADE, "impressao_dobras": IMPRESSAO_DOBRAS}, PASTA / "config_busca.json")
else: RUN_ID = PASTA.name.replace("svm_busca_", "")
for sub in ("fits", "modelos", "dados"): (PASTA / sub).mkdir(exist_ok=True)
for nome, arr in (("X_aj", X_aj), ("y_aj", y_aj), ("X_va", X_va)):                       # os dados ficam em disco: cada processo os lê por memória mapeada (nada é copiado pelo canal de comunicação)
    if not (PASTA / "dados" / f"{nome}.npy").exists(): np.save(PASTA / "dados" / f"{nome}.npy", arr)
print(f"\nPasta da execução: {PASTA}" + ("  (RETOMANDO uma busca interrompida)" if RETOMANDO else ""))


Atributos do modelo: 44 (42 + 2 indicadores) | retiradas na ablação: ['bwd_packets_s', 'inbound', 'total_backward_packets']

Ajuste: 239,671 linhas | validação: 59,918 | refit: 299,589 | 7 classes


,BENIGN,LDAP,MSSQL,NetBIOS,Syn,UDP,UDPLag
ajuste,40000,24079,40000,15592,40000,40000,40000
validação,10000,6020,10000,3898,10000,10000,10000


Dobras: 5, estratificadas por classe (~47,934 linhas de avaliação cada) | impressão digital das dobras: 7e97c47f
✔ impressão digital das dobras do RF (07m): 7e97c47f → IGUAIS: RF e SVM podem ser comparados dobra a dobra
gamma de referência (scale) = 0.02379 | gammas da grade: 0.25× = 0.00595, 1× = 0.02379, 4× = 0.09515 (os mesmos valores absolutos valem no refit)

Pasta da execução: C:\Users\Bruno\downloads\tcc_codigo\railway-ch\artefatos_modelos\svm_multiclasse_busca\svm_busca_20261107_013148


## 3. A busca (a parte longa)
Os 54 treinos (9 configurações × 5 dobras + 1 treino completo) rodam em **processos à parte**, com **vários ao mesmo tempo só se houver memória livre** (veja a introdução). Cada treino é salvo ao terminar: **se interromper, rode a célula de novo**. O tempo de cada sessão e o ganho observado do paralelismo ficam em `tempos_busca.csv`.

In [8]:
# ---------- 3.1 Busca ----------
def arq_fit(C, f, tipo, k=0): return PASTA / "fits" / (f"{nome_cfg(C, f)}_dobra{k}.npz" if tipo == "dobra" else f"{nome_cfg(C, f)}_completo.npz")
def salvar_fit(C, f, tipo, k, meta, sc):
    arq = arq_fit(C, f, tipo, k); tmp = arq.with_name(arq.stem + ".tmp.npz"); np.savez_compressed(tmp, scores=sc, meta=json.dumps(_limpar(meta))); os.replace(tmp, arq)      # o modelo (se houver) já foi gravado ANTES, pelo processo
def marcar_limite(C, f, tipo, k): salvar_fit(C, f, tipo, k, {"passou_limite": True, "limite_s": LIMITE_FIT_S, "C": C, "gamma_fator": f}, np.zeros((0, K), np.float32))
def feito(C, f, tipo, k):
    a = arq_fit(C, f, tipo, k)
    if not a.exists(): return False
    try: m = json.loads(str(np.load(a)["meta"]))
    except Exception: return False
    return not (m.get("passou_limite") and LIMITE_FIT_S > float(m.get("limite_s", 0)))        # um marcador antigo é refeito se o limite foi aumentado
def lancar_tarefa(t):
    C, f, tipo, k = t; ex = LokyPool(max_workers=1); d = str(PASTA / "dados")
    if tipo == "dobra": fut = ex.submit(_worker_dobra, d, DOBRAS[k][0], DOBRAS[k][1], C, GAMAS[f], None, None)
    else: fut = ex.submit(_worker_dobra, d, None, None, C, GAMAS[f], str(PASTA / "modelos" / f"{nome_cfg(C, f)}.joblib"), None)
    return ex, fut
TAREFAS = [(C, f, tipo, k) for (C, f) in CONFIGS for tipo, k in [("dobra", i) for i in range(K_CV)] + ([("completo", 0)] if AVALIAR_VALIDACAO_POR_CONFIG else [])]
pendentes = [t for t in TAREFAS if not feito(*t)]
rot_t = lambda t: f"{rot_cfg(t[0], t[1])} | " + (f"dobra {t[3] + 1}/{K_CV}" if t[2] == "dobra" else "treino completo")
print(f"Treinos prontos: {len(TAREFAS) - len(pendentes)}/{len(TAREFAS)}" + ("  (retomando)" if len(TAREFAS) - len(pendentes) else "") + f" | a treinar agora: {len(pendentes)}")
STATS_BUSCA = {}
if pendentes:
    K_PAR = processos_seguros(PROCESSOS_MAX, RAM_POR_PROCESSO_GIB); livre0 = ram_livre_gib()
    print(f"Memória livre agora: {livre0:.1f} GiB | cada treino precisa de ~{RAM_POR_PROCESSO_GIB:.1f} GiB × 1,3 e a reserva é {RAM_RESERVA_GIB:.1f} GiB → posso começar com {K_PAR} treino(s) simultâneo(s) (teto {PROCESSOS_MAX}; o escalonador reavalia a memória a cada partida)")
    if K_PAR < PROCESSOS_MAX: print(f"  ⚠ a memória livre só comporta {K_PAR} treino(s) ao mesmo tempo. Fechar o Meet, o Edge e os outros notebooks libera memória e deixa a busca mais rápida.")
    barra = Barra(len(TAREFAS), "Busca de hiperparâmetros (SVM multiclasse)"); barra.n = len(TAREFAS) - len(pendentes); barra._mostrar(forcar=True); novas, soma_treino = 0, 0.0
    t_sessao, inicio_sessao, interrompida = time.perf_counter(), datetime.now().isoformat(timespec="seconds"), True
    def coletar(t, res, k_ini):
        global novas, soma_treino
        C, f, tipo, k = t
        if res is None:
            marcar_limite(C, f, tipo, k); barra.avancar(1, f"{rot_t(t)}: PASSOU DO LIMITE ({fmt_tempo(LIMITE_FIT_S)}) e foi encerrado"); print(f"  ⚠ {rot_t(t)} passou de {fmt_tempo(LIMITE_FIT_S)}: a configuração fica fora da escolha")
        else:
            meta, sc = res; meta["simultaneos_na_partida"] = int(k_ini); salvar_fit(C, f, tipo, k, meta, sc); soma_treino += meta["tempo_treino_s"]
            barra.avancar(1, f"{rot_t(t)}: {fmt_tempo(meta['tempo_treino_s'])}, {meta['n_sv']:,} vetores de suporte ({k_ini} simultâneo(s) na partida)")
        novas += 1
    try:
        STATS_BUSCA = rodar_tarefas(pendentes, lancar_tarefa, coletar, barra, LIMITE_FIT_S, PROCESSOS_MAX, RAM_POR_PROCESSO_GIB, "busca", PARAR_APOS_FITS)       # k_max = o teto; a memória livre decide, a cada partida, quantos rodam de fato
        interrompida = False
    except KeyboardInterrupt:
        print(f"\nInterrompido. Os treinos prontos ficam salvos ({sum(feito(*t) for t in TAREFAS)}/{len(TAREFAS)}): rode esta célula de novo para retomar."); raise
    finally:                                                         # grava o tempo desta sessão mesmo se a busca for interrompida ou falhar
        rel = time.perf_counter() - t_sessao
        rec = {"inicio": inicio_sessao, "fim": datetime.now().isoformat(timespec="seconds"), "relogio_s": rel, "treinos_novos": novas, "soma_tempos_treino_s": soma_treino, "ganho_paralelismo": (soma_treino / rel) if rel > 0 and novas else float("nan"),
               "max_simultaneos": STATS_BUSCA.get("max_simultaneos"), "interrompidos_por_memoria": STATS_BUSCA.get("interrompidos_por_memoria"), "interrompida": interrompida}
        pd.DataFrame([rec]).to_csv(PASTA / "tempos_busca.csv", mode="a", header=not (PASTA / "tempos_busca.csv").exists(), index=False)
    barra.fechar(); print(f"✔ Busca concluída: {len(TAREFAS)} treinos em {PASTA / 'fits'} | tempo desta sessão: {fmt_tempo(rec['relogio_s'])} | treinos simultâneos no máximo: {STATS_BUSCA.get('max_simultaneos')} | "
                          f"ganho observado do paralelismo: {rec['ganho_paralelismo']:.2f}× (soma dos tempos de treino ÷ relógio)" + (f" | treinos interrompidos por falta de memória: {STATS_BUSCA['interrompidos_por_memoria']}" if STATS_BUSCA.get("interrompidos_por_memoria") else ""))

def garantir_completo(C, f):
    # O treino completo (ajuste inteiro → validação) de uma configuração: usa o gravado ou, se não existir (AVALIAR_VALIDACAO_POR_CONFIG = False), treina agora.
    if not feito(C, f, "completo", 0):
        rotulo = f"{rot_cfg(C, f)} | treino completo"; ex, fut = lancar_tarefa((C, f, "completo", 0)); t0 = time.perf_counter()
        while not fut.done():
            if time.perf_counter() - t0 > LIMITE_FIT_S: ex.shutdown(wait=True, kill_workers=True); raise AssertionError(f"o treino completo de {rot_cfg(C, f)} passou do limite de tempo ({fmt_tempo(LIMITE_FIT_S)}): aumente LIMITE_FIT_S")
            time.sleep(0.5)
        meta, sc = fut.result(); ex.shutdown(wait=True); salvar_fit(C, f, "completo", 0, meta, sc)
    z = np.load(arq_fit(C, f, "completo", 0)); return z["scores"], json.loads(str(z["meta"]))


Treinos prontos: 0/54 | a treinar agora: 54
Memória livre agora: 0.2 GiB | cada treino precisa de ~1.0 GiB × 1,3 e a reserva é 1.0 GiB → posso começar com 1 treino(s) simultâneo(s) (teto 4; o escalonador reavalia a memória a cada partida)
  ⚠ a memória livre só comporta 1 treino(s) ao mesmo tempo. Fechar o Meet, o Edge e os outros notebooks libera memória e deixa a busca mais rápida.


Busca de hiperparâmetros (SVM multiclasse): concluído em 28h 13min
✔ Busca concluída: 54 treinos em C:\Users\Bruno\downloads\tcc_codigo\railway-ch\artefatos_modelos\svm_multiclasse_busca\svm_busca_20261107_013148\fits | tempo desta sessão: 28h 13min | treinos simultâneos no máximo: 1 | ganho observado do paralelismo: 0.75× (soma dos tempos de treino ÷ relógio)


## 4. Métricas de cada configuração
Calcula, para cada configuração: as métricas multiclasse em cada dobra (fora da dobra) e na validação (sem pesos e na proporção do treino), o recall por classe e os custos. A **média das dobras** é o valor da validação cruzada, e o **erro-padrão** é o desvio entre dobras ÷ √5.

In [9]:
# ---------- 4.1 Métricas por configuração ----------
def ler_fit(C, f, tipo, k=0):
    z = np.load(arq_fit(C, f, tipo, k)); return z["scores"], json.loads(str(z["meta"]))
M_BASE = ("roc_auc", "acuracia_balanceada", "acuracia", "precisao_macro", "f1_macro", "f1_ponderado", "mcc")
M_PROP = ("acuracia", "precisao_macro", "f1_macro", "f1_ponderado", "mcc")
linhas_fold, linhas_val, linhas_cls, linhas_custo, incompletas, CFG_OK = [], [], [], [], [], []
barra = Barra(len(CONFIGS), "Calculando as métricas")
for (C, f) in CONFIGS:
    fits = [ler_fit(C, f, "dobra", k) for k in range(K_CV)]
    if any(m.get("passou_limite") for _, m in fits): incompletas.append(rot_cfg(C, f)); barra.avancar(1); continue
    completo = ler_fit(C, f, "completo", 0) if (AVALIAR_VALIDACAO_POR_CONFIG and feito(C, f, "completo", 0)) else None
    if completo is not None and completo[1].get("passou_limite"): completo = None
    oof = np.full((len(y_aj), K), np.nan, np.float32)
    for k, (sc, m) in enumerate(fits): oof[DOBRAS[k][1]] = sc
    assert not np.isnan(oof).any(), f"previsões fora da dobra incompletas em {rot_cfg(C, f)}"; CFG_OK.append((C, f))
    base = {"C": C, "gamma_fator": f, "gamma": GAMAS[f], "config": rot_cfg(C, f)}
    for k, (_, ite) in enumerate(DOBRAS, 1):
        a, _ = met(y_aj[ite], oof[ite]); linhas_fold.append({**base, "dobra": k, **{m: a[m] for m in M_BASE}, "n_sv": fits[k - 1][1]["n_sv"], "tempo_treino_s": fits[k - 1][1]["tempo_treino_s"]})
    for c, r in zip(CLASSES, recall_por_classe(y_aj, oof)): linhas_cls.append({**base, "fonte": "cv_fora_da_dobra", "classe": c, "recall": r})
    custo = {**base, "n_sv_cv": float(np.mean([m["n_sv"] for _, m in fits])), "t_treino_cv_s": float(np.mean([m["tempo_treino_s"] for _, m in fits])), "pico_ram_mib": float(max(m["pico_ram_mib"] for _, m in fits))}
    if completo is not None:
        val, mc = completo; a, _ = met(y_va, val); b, _ = met(y_va, val, w_va)
        linhas_val.append({**base, **{m: a[m] for m in M_BASE}, **{m + "_prop": b[m] for m in M_PROP}})
        for c, r in zip(CLASSES, recall_por_classe(y_va, val)): linhas_cls.append({**base, "fonte": "validacao", "classe": c, "recall": r})
        mod = PASTA / "modelos" / f"{nome_cfg(C, f)}.joblib"
        custo.update({"n_sv": mc["n_sv"], "t_treino_completo_s": mc["tempo_treino_s"], "cpu_completo_s": mc["cpu_s"], "t_prev_val_s": mc["tempo_previsao_s"], "ns_por_par": mc["ns_por_par"], "modelo_mib": os.path.getsize(mod) / 2 ** 20 if mod.exists() else float("nan"),
                      "pico_ram_mib": max(custo["pico_ram_mib"], mc["pico_ram_mib"])})
    else: custo.update({"n_sv": custo["n_sv_cv"], "t_treino_completo_s": float("nan"), "cpu_completo_s": float("nan"), "t_prev_val_s": float("nan"), "ns_por_par": float("nan"), "modelo_mib": float("nan")})
    linhas_custo.append(custo); barra.avancar(1)
barra.fechar()
if incompletas: print(f"⚠ {len(incompletas)} configuração(ões) passaram do limite de tempo e ficam FORA da escolha: {incompletas}")
assert CFG_OK, "nenhuma configuração concluiu todas as dobras"
df_fold, df_val, df_cls, df_custo = (pd.DataFrame(x) for x in (linhas_fold, linhas_val, linhas_cls, linhas_custo)); CHAVE = ["C", "gamma_fator"]

def resumo_cv(metrica):
    g = df_fold.groupby(CHAVE)[metrica]; return pd.DataFrame({f"cv_{metrica}": g.mean(), f"cv_{metrica}_ep": g.std(ddof=1) / np.sqrt(K_CV)})        # média e erro-padrão entre as dobras
M_CV = ["roc_auc", "acuracia_balanceada", "acuracia", "f1_macro", "mcc"]
df_cfg = pd.concat([resumo_cv(m) for m in M_CV], axis=1)
if len(df_val):
    M_VAL = ["roc_auc", "acuracia_balanceada", "acuracia", "f1_macro", "mcc", "f1_macro_prop", "mcc_prop"]
    df_cfg = df_cfg.join(df_val.set_index(CHAVE)[M_VAL].add_prefix("val_"))
df_cfg = df_cfg.reset_index().merge(df_custo.drop(columns=["gamma", "config"]), on=CHAVE); df_cfg.insert(0, "config", [rot_cfg(c, f) for c, f in zip(df_cfg.C, df_cfg.gamma_fator)]); df_cfg.insert(3, "gamma", [GAMAS[f] for f in df_cfg.gamma_fator])
tempos_sessoes = pd.read_csv(PASTA / "tempos_busca.csv") if (PASTA / "tempos_busca.csv").exists() else pd.DataFrame(columns=["relogio_s"])
TEMPO_BUSCA_RELOGIO = float(tempos_sessoes["relogio_s"].sum()) if len(tempos_sessoes) else float("nan")
N_SVMS = len(df_fold) + (int(df_cfg["t_treino_completo_s"].notna().sum()) if "t_treino_completo_s" in df_cfg else 0)
TEMPO_TREINO_SOMADO = float(df_fold.tempo_treino_s.sum() + (df_cfg["t_treino_completo_s"].dropna().sum() if "t_treino_completo_s" in df_cfg else 0.0))
print(f"\n⏱ Tempo da busca: {fmt_tempo(TEMPO_BUSCA_RELOGIO)} de relógio em {len(tempos_sessoes)} sessão(ões) | soma dos tempos de treino dos {N_SVMS} SVMs: {fmt_tempo(TEMPO_TREINO_SOMADO)}" + (f" | ganho médio do paralelismo: {TEMPO_TREINO_SOMADO / TEMPO_BUSCA_RELOGIO:.2f}×" if TEMPO_BUSCA_RELOGIO == TEMPO_BUSCA_RELOGIO and TEMPO_BUSCA_RELOGIO > 0 else ""))
print(f"✔ {len(df_cfg)} configurações | {len(df_fold):,} medidas por dobra | {len(df_val)} medidas na validação")
cols_ver = [c for c in ("config", "cv_roc_auc", "cv_roc_auc_ep", "cv_mcc", "cv_mcc_ep", "cv_f1_macro", "cv_acuracia_balanceada", "val_roc_auc", "val_mcc", "val_f1_macro", "n_sv", "t_treino_completo_s", "t_treino_cv_s") if c in df_cfg.columns]
print("\n== Todas as configurações (CV: média das dobras; ep = erro-padrão entre dobras; val = validação separada) ==")
display(df_cfg[cols_ver].round(5).reset_index(drop=True))


Calculando as métricas: concluído em 9s

⏱ Tempo da busca: 28h 13min de relógio em 1 sessão(ões) | soma dos tempos de treino dos 54 SVMs: 21h 16min | ganho médio do paralelismo: 0.75×
✔ 9 configurações | 45 medidas por dobra | 9 medidas na validação

== Todas as configurações (CV: média das dobras; ep = erro-padrão entre dobras; val = validação separada) ==


,config,cv_roc_auc,cv_roc_auc_ep,cv_mcc,cv_mcc_ep,cv_f1_macro,cv_acuracia_balanceada,val_roc_auc,val_mcc,val_f1_macro,n_sv,t_treino_completo_s,t_treino_cv_s
0,C=1|gamma=0.25x,0.97050,0.00031,0.86490,0.00066,0.85927,0.85682,0.97171,0.86602,0.86114,75238,2620.27100,1135.55724
1,C=1|gamma=1x,0.97523,0.00016,0.87135,0.00051,0.86475,0.86085,0.97572,0.87212,0.86621,68351,3033.14713,996.75793
2,C=1|gamma=4x,0.97545,0.00011,0.87805,0.00045,0.87026,0.86515,0.97557,0.87929,0.87195,65940,3650.47191,1299.80733
3,C=10|gamma=0.25x,0.97203,0.00013,0.86949,0.00082,0.86354,0.85943,0.97260,0.87067,0.86505,67532,2578.01186,996.07204
4,C=10|gamma=1x,0.97247,0.00017,0.87760,0.00074,0.86976,0.86502,0.97256,0.87918,0.87163,64746,2536.09700,961.91891
5,C=10|gamma=4x,0.97399,0.00009,0.88016,0.00060,0.87274,0.86743,0.97428,0.88161,0.87468,64149,3461.48329,1385.38036
6,C=100|gamma=0.25x,0.96921,0.00036,0.87578,0.00041,0.86818,0.86364,0.96854,0.87683,0.86963,64629,2342.27604,989.46918
7,C=100|gamma=1x,0.96969,0.00014,0.87966,0.00046,0.87208,0.86693,0.97019,0.88071,0.87360,62867,1811.78719,1063.39298
8,C=100|gamma=4x,0.97334,0.00014,0.88105,0.00047,0.87373,0.86908,0.97355,0.88245,0.87566,62797,2730.10989,1531.89361


## 5. Escolha da combinação (a mesma regra do RF)
(1) acha a **melhor** configuração em cada métrica de decisão (ROC-AUC um-contra-resto e MCC); (2) toda configuração a menos de **1 erro-padrão** da melhor **empata**; (3) fica elegível quem empata **nas duas**; (4) entre as elegíveis vence a **mais barata** (menos vetores de suporte). Se nenhuma for elegível, vale a melhor posição média (plano B, avisado).

In [10]:
# ---------- 5.1 Escolha ----------
def escolher(df, metricas_dec=METRICAS_DECISAO, mult=MULT_ERRO_PADRAO):
    d = df.reset_index(drop=True).copy(); elegivel = np.ones(len(d), bool); info = {}
    for m in metricas_dec:
        i = int(d[f"cv_{m}"].idxmax()); lim = d.loc[i, f"cv_{m}"] - mult * d.loc[i, f"cv_{m}_ep"]
        d[f"empata_{m}"] = d[f"cv_{m}"] >= lim; elegivel &= d[f"empata_{m}"].to_numpy()
        info[m] = {"melhor": d.loc[i, "config"], "valor_melhor": float(d.loc[i, f"cv_{m}"]), "erro_padrao_melhor": float(d.loc[i, f"cv_{m}_ep"]), "limite": float(lim), "n_empatadas": int(d[f"empata_{m}"].sum())}
    fallback = not elegivel.any()
    if fallback:
        rk = sum(d[f"cv_{m}"].rank(ascending=False) for m in metricas_dec); elegivel = (rk == rk.min()).to_numpy()
    cand = d[elegivel].sort_values(["n_sv", "t_treino_cv_s", "C"], ascending=[True, True, True]).reset_index(drop=True)
    return cand.iloc[0], cand, info, fallback

ESC, CAND, INFO_SEL, FALLBACK = escolher(df_cfg)
C_ESC, F_ESC = float(ESC["C"]), float(ESC["gamma_fator"]); NOME_ESC = nome_cfg(C_ESC, F_ESC)
HIPERPARAMETROS_FINAL = dict(kernel="rbf", C=C_ESC, gamma=GAMAS[F_ESC], gamma_fator=F_ESC, cache_size=CACHE_MB, class_weight="balanced", normalizador="StandardScaler nas colunas numéricas (média e desvio do próprio treino)")
ESCOLHEU_PADRAO = (C_ESC, F_ESC) == (PADRAO["C"], PADRAO["gamma_fator"])
print("== Passo 1: melhor configuração em cada métrica de decisão e o limite de empate (melhor − 1 erro-padrão) ==")
display(pd.DataFrame(INFO_SEL).T.rename(columns={"melhor": "melhor configuração", "valor_melhor": "valor", "erro_padrao_melhor": "erro-padrão", "limite": "limite de empate", "n_empatadas": "configs que empatam"}))
print(f"\n== Passo 2: elegíveis (empatam nas duas métricas): {len(CAND)} de {len(df_cfg)} ==" + ("   ⚠ NENHUMA empatou nas duas: usei a melhor posição média (plano B)" if FALLBACK else ""))
cols = [c for c in ("config", "cv_roc_auc", "cv_roc_auc_ep", "cv_mcc", "cv_mcc_ep", "cv_f1_macro", "val_roc_auc", "val_mcc", "val_f1_macro", "n_sv", "modelo_mib", "t_treino_cv_s", "t_treino_completo_s") if c in CAND.columns]
display(CAND[cols].round(5).head(15))
print(f"\n✔ ESCOLHIDA: C = {C_ESC:g}, gamma = {F_ESC:g}× o de referência ({GAMAS[F_ESC]:.5f})" + ("  (é o padrão do scikit-learn)" if ESCOLHEU_PADRAO else "") + f"  (a mais barata entre as {len(CAND)} elegíveis: {ESC['n_sv']:,.0f} vetores de suporte)")

def _linha(rotulo, r): return {"configuração": rotulo, "config": r["config"], **{c: r[c] for c in ("cv_roc_auc", "cv_roc_auc_ep", "cv_mcc", "cv_mcc_ep", "cv_f1_macro", "cv_acuracia_balanceada", "val_roc_auc", "val_mcc", "val_f1_macro", "n_sv", "modelo_mib", "t_treino_cv_s", "t_treino_completo_s", "t_prev_val_s") if c in r.index}}
comp = [_linha("escolhida", ESC)]
pd_ = df_cfg[(df_cfg.C == PADRAO["C"]) & (df_cfg.gamma_fator == PADRAO["gamma_fator"])]
if len(pd_): comp.append(_linha("padrão (C = 1, gamma = scale)", pd_.iloc[0]))
for m in METRICAS_DECISAO: comp.append(_linha(f"melhor em {m} (CV)", df_cfg.loc[df_cfg[f"cv_{m}"].idxmax()]))
print("\n== A escolhida × o padrão × as melhores de cada métrica ==")
tab_comp = pd.DataFrame(comp); display(tab_comp.round(5))
if "val_roc_auc" in df_cfg.columns:
    rk_val = {m: int((df_cfg[f"val_{m}"] > ESC[f"val_{m}"]).sum()) + 1 for m in ("roc_auc", "mcc")}
    print(f"Conferência na validação separada (não decide nada): a escolhida ficou em {rk_val['roc_auc']}º de {len(df_cfg)} em ROC-AUC e em {rk_val['mcc']}º em MCC.")


== Passo 1: melhor configuração em cada métrica de decisão e o limite de empate (melhor − 1 erro-padrão) ==


,melhor configuração,valor,erro-padrão,limite de empate,configs que empatam
roc_auc,C=1|gamma=4x,0.975455,0.000109,0.975345,1
mcc,C=100|gamma=4x,0.88105,0.000474,0.880576,1



== Passo 2: elegíveis (empatam nas duas métricas): 3 de 9 ==   ⚠ NENHUMA empatou nas duas: usei a melhor posição média (plano B)


,config,cv_roc_auc,cv_roc_auc_ep,cv_mcc,cv_mcc_ep,cv_f1_macro,val_roc_auc,val_mcc,val_f1_macro,n_sv,modelo_mib,t_treino_cv_s,t_treino_completo_s
0,C=100|gamma=4x,0.97334,0.00014,0.88105,0.00047,0.87373,0.97355,0.88245,0.87566,62797,4.00349,1531.89361,2730.10989
1,C=10|gamma=4x,0.97399,0.00009,0.88016,0.00060,0.87274,0.97428,0.88161,0.87468,64149,4.10241,1385.38036,3461.48329
2,C=1|gamma=4x,0.97545,0.00011,0.87805,0.00045,0.87026,0.97557,0.87929,0.87195,65940,4.23604,1299.80733,3650.47191



✔ ESCOLHIDA: C = 100, gamma = 4× o de referência (0.09515)  (a mais barata entre as 3 elegíveis: 62,797 vetores de suporte)

== A escolhida × o padrão × as melhores de cada métrica ==


,configuração,config,cv_roc_auc,cv_roc_auc_ep,cv_mcc,cv_mcc_ep,cv_f1_macro,cv_acuracia_balanceada,val_roc_auc,val_mcc,val_f1_macro,n_sv,modelo_mib,t_treino_cv_s,t_treino_completo_s,t_prev_val_s
0,escolhida,C=100|gamma=4x,0.97334,0.00014,0.88105,0.00047,0.87373,0.86908,0.97355,0.88245,0.87566,62797,4.00349,1531.89361,2730.10989,522.44144
1,"padrão (C = 1, gamma = scale)",C=1|gamma=1x,0.97523,0.00016,0.87135,0.00051,0.86475,0.86085,0.97572,0.87212,0.86621,68351,4.06263,996.75793,3033.14713,684.58511
2,melhor em roc_auc (CV),C=1|gamma=4x,0.97545,0.00011,0.87805,0.00045,0.87026,0.86515,0.97557,0.87929,0.87195,65940,4.23604,1299.80733,3650.47191,804.26110
3,melhor em mcc (CV),C=100|gamma=4x,0.97334,0.00014,0.88105,0.00047,0.87373,0.86908,0.97355,0.88245,0.87566,62797,4.00349,1531.89361,2730.10989,522.44144


Conferência na validação separada (não decide nada): a escolhida ficou em 4º de 9 em ROC-AUC e em 1º em MCC.


## 6. Quanto cada hiperparâmetro influencia
Efeito principal, tamanho do efeito comparado com o ruído, um de cada vez a partir do padrão, decomposição da variação (C, gamma e a interação) e gráficos (efeitos principais e mapas de calor C × gamma).

In [11]:
# ---------- 6.1 Influência dos hiperparâmetros ----------
INFL = [("ROC-AUC (CV)", "cv_roc_auc"), ("MCC (CV)", "cv_mcc"), ("F1 macro (CV)", "cv_f1_macro"), ("acurácia balanceada (CV)", "cv_acuracia_balanceada"),
        ("ROC-AUC (validação)", "val_roc_auc"), ("MCC (validação)", "val_mcc"), ("F1 macro (validação)", "val_f1_macro"),
        ("vetores de suporte", "n_sv"), ("tamanho do modelo (MiB)", "modelo_mib"), ("tempo de treino completo (s)", "t_treino_completo_s"), ("tempo de treino nas dobras (s)", "t_treino_cv_s"), ("tempo de previsão na validação (s)", "t_prev_val_s")]
INFL = [(r, c) for r, c in INFL if c in df_cfg.columns and df_cfg[c].notna().any()]
NIVEIS = {"C": [float(c) for c in GRADE["C"]], "gamma_fator": [float(f) for f in GRADE["gamma_fator"]]}; NOME_HP = {"C": "C", "gamma_fator": "gamma (× referência)"}
def ruido(col):
    return float(df_cfg[f"{col}_ep"].mean()) if (col.startswith("cv_") and f"{col}_ep" in df_cfg) else float("nan")

linhas_ef, linhas_amp = [], []
for fator, niveis in NIVEIS.items():
    for nv in niveis:
        sub = df_cfg[df_cfg[fator] == nv]; linhas_ef.append({"hiperparametro": NOME_HP[fator], "valor": nv, **{r: sub[c].mean() for r, c in INFL}})
tab_ef = pd.DataFrame(linhas_ef)
for fator in NIVEIS:
    sub = tab_ef[tab_ef.hiperparametro == NOME_HP[fator]]
    for r, c in INFL:
        amp = float(sub[r].max() - sub[r].min()); rz = ruido(c)
        leitura = ("sem ruído de referência" if c.startswith("val_") else "efeito no custo") if rz != rz else ("ACIMA do ruído" if amp > 2 * rz else "dentro do ruído")
        linhas_amp.append({"hiperparametro": NOME_HP[fator], "métrica": r, "amplitude": amp, "ruído (erro-padrão)": rz, "amplitude / ruído": amp / rz if rz == rz and rz > 0 else float("nan"), "leitura": leitura})
tab_amp = pd.DataFrame(linhas_amp)
print("== Efeito principal: média da métrica em cada valor do hiperparâmetro (média sobre os valores do outro) ==")
display(tab_ef.round(5))
print("\n== Tamanho do efeito: amplitude (maior média − menor média) comparada com o ruído ==")
display(tab_amp.pivot(index="métrica", columns="hiperparametro", values="amplitude").loc[[r for r, _ in INFL]].round(5))
display(tab_amp[tab_amp["métrica"].isin(["ROC-AUC (CV)", "MCC (CV)"])].round(4))

# um de cada vez, a partir do padrão
base = df_cfg[(df_cfg.C == PADRAO["C"]) & (df_cfg.gamma_fator == PADRAO["gamma_fator"])]; tab_um = pd.DataFrame()
if len(base):
    b0 = base.iloc[0]; linhas_um = []
    for _, r in df_cfg.iterrows():
        difs = [f for f in NIVEIS if r[f] != b0[f]]
        if len(difs) == 1:
            f = difs[0]; linhas_um.append({"muda": NOME_HP[f], "de": b0[f], "para": r[f], **{f"Δ {n}": r[c] - b0[c] for n, c in INFL}, "Δ ROC-AUC (CV) em erros-padrão": (r["cv_roc_auc"] - b0["cv_roc_auc"]) / b0["cv_roc_auc_ep"],
                                           "Δ MCC (CV) em erros-padrão": (r["cv_mcc"] - b0["cv_mcc"]) / b0["cv_mcc_ep"]})
    tab_um = pd.DataFrame(linhas_um); print("\n== Um de cada vez, a partir do padrão (C = 1, gamma = 1× o de referência) ==")
    display(tab_um[[c for c in ("muda", "de", "para", "Δ ROC-AUC (CV)", "Δ MCC (CV)", "Δ ROC-AUC (CV) em erros-padrão", "Δ MCC (CV) em erros-padrão", "Δ vetores de suporte", "Δ tempo de treino nas dobras (s)") if c in tab_um.columns]].round(5))
else: print("\n(o padrão não está na grade: tabela 'um de cada vez' omitida)")

# decomposição da variação (dois fatores, uma medida por célula: a interação é o resíduo)
def decompor(Y):
    a, b = Y.shape; mu = Y.mean(); ma, mb = Y.mean(1), Y.mean(0)
    ss = {"C": b * ((ma - mu) ** 2).sum(), "gamma (× referência)": a * ((mb - mu) ** 2).sum(), "interação C × gamma (resíduo)": ((Y - ma[:, None] - mb[None, :] + mu) ** 2).sum()}; tot = ((Y - mu) ** 2).sum()
    return {k: (float(v / tot) if tot > 0 else float("nan")) for k, v in ss.items()}
def cubo(col):
    Y = np.full((len(NIVEIS["C"]), len(NIVEIS["gamma_fator"])), np.nan)
    for i, c in enumerate(NIVEIS["C"]):
        for j, f in enumerate(NIVEIS["gamma_fator"]):
            s = df_cfg[(df_cfg.C == c) & (df_cfg.gamma_fator == f)][col]; Y[i, j] = float(s.iloc[0]) if len(s) else np.nan
    return Y
COMPLETA = len(df_cfg) == len(NIVEIS["C"]) * len(NIVEIS["gamma_fator"])
if COMPLETA:
    decomp = pd.DataFrame({r: decompor(cubo(c)) for r, c in INFL}); decomp.index.name = "fonte da variação"
    print("\n== Decomposição da variação entre as configurações (fração de cada fonte; cada coluna soma 1) ==")
    display((decomp[[c for c in ("ROC-AUC (CV)", "MCC (CV)", "ROC-AUC (validação)", "vetores de suporte", "tempo de treino completo (s)") if c in decomp.columns]] * 100).round(1).add_suffix(" (%)"))
else:
    decomp = pd.DataFrame({"sem dados": {"grade incompleta": float("nan")}}); decomp.index.name = "fonte da variação"; print("\n(grade incompleta por causa de configurações que passaram do limite de tempo: decomposição omitida)")

# leitura automática
print("\n== Leitura automática (ROC-AUC e MCC da validação cruzada) ==")
for fator in NIVEIS:
    for r in ("ROC-AUC (CV)", "MCC (CV)"):
        t = tab_amp[(tab_amp.hiperparametro == NOME_HP[fator]) & (tab_amp["métrica"] == r)].iloc[0]; sub = tab_ef[tab_ef.hiperparametro == NOME_HP[fator]]
        print(f"  {NOME_HP[fator]:21s} {r:25s} amplitude {t['amplitude']:.5f} ({t['amplitude / ruído']:.1f}× o ruído) → {t['leitura']}; melhor valor na média: {sub.loc[sub[r].idxmax(), 'valor']:g}")

# gráficos
try:
    import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
    fig, ax = plt.subplots(2, 2, figsize=(9, 6.2))
    for j, fator in enumerate(NIVEIS):
        sub = tab_ef[tab_ef.hiperparametro == NOME_HP[fator]]
        for i, (r, c) in enumerate((("ROC-AUC (CV)", "cv_roc_auc"), ("MCC (CV)", "cv_mcc"))):
            ax[i, j].errorbar(range(len(sub)), sub[r], yerr=ruido(c), marker="o", capsize=4); ax[i, j].set_xticks(range(len(sub))); ax[i, j].set_xticklabels([f"{v:g}" for v in sub["valor"]]); ax[i, j].grid(alpha=.3)
            ax[i, j].set_title(NOME_HP[fator] if i == 0 else ""); ax[i, j].set_ylabel(r if j == 0 else ""); ax[i, j].set_xlabel(NOME_HP[fator] if i == 1 else "")
    fig.suptitle("Efeito principal de cada hiperparâmetro (média sobre o outro; barra = 1 erro-padrão típico)"); fig.tight_layout(); fig.savefig(PASTA / "influencia_efeitos_principais.png", dpi=130); plt.close(fig)
    fig, ax = plt.subplots(1, 3, figsize=(13, 4))
    for a_, (r, c) in zip(ax, (("ROC-AUC (CV)", "cv_roc_auc"), ("MCC (CV)", "cv_mcc"), ("vetores de suporte", "n_sv"))):
        pv = df_cfg.pivot_table(index="C", columns="gamma_fator", values=c, aggfunc="mean").reindex(index=NIVEIS["C"], columns=NIVEIS["gamma_fator"])
        im = a_.imshow(pv.values, aspect="auto", cmap="viridis"); a_.set_xticks(range(pv.shape[1])); a_.set_xticklabels([f"{v:g}×" for v in pv.columns]); a_.set_yticks(range(pv.shape[0])); a_.set_yticklabels([f"{v:g}" for v in pv.index])
        a_.set_xlabel("gamma (× o de referência)"); a_.set_ylabel("C"); a_.set_title(r); plt.colorbar(im, ax=a_)
        for (ii, jj), v in np.ndenumerate(pv.values): a_.text(jj, ii, f"{v:.4g}", ha="center", va="center", color="w", fontsize=8)
    fig.suptitle("Interação C × gamma"); fig.tight_layout(); fig.savefig(PASTA / "influencia_interacao.png", dpi=130); plt.close(fig); print("\n✔ gráficos salvos: influencia_efeitos_principais.png, influencia_interacao.png")
except ImportError:
    print("\n(matplotlib não está instalado: gráficos não gerados)")

== Efeito principal: média da métrica em cada valor do hiperparâmetro (média sobre os valores do outro) ==


,hiperparametro,valor,ROC-AUC (CV),MCC (CV),F1 macro (CV),acurácia balanceada (CV),ROC-AUC (validação),MCC (validação),F1 macro (validação),vetores de suporte,tamanho do modelo (MiB),tempo de treino completo (s),tempo de treino nas dobras (s),tempo de previsão na validação (s)
0,C,1.00,0.97373,0.87143,0.86476,0.86094,0.97433,0.87248,0.86643,69843.00000,4.24689,3101.29668,1144.04083,712.98367
1,C,10.00,0.97283,0.87575,0.86868,0.86396,0.97315,0.87715,0.87045,65475.66667,3.90553,2858.53072,1114.45710,557.03660
2,C,100.00,0.97075,0.87883,0.87133,0.86655,0.97076,0.88000,0.87296,63431.00000,3.73375,2294.72437,1194.91859,650.42479
3,gamma (× referência),0.25,0.97058,0.87006,0.86366,0.85997,0.97095,0.87117,0.86527,69133.00000,3.95869,2513.51963,1040.36615,640.48456
4,gamma (× referência),1.00,0.97246,0.87620,0.86887,0.86427,0.97282,0.87734,0.87048,65321.33333,3.81351,2460.34378,1007.35660,656.60778
5,gamma (× referência),4.00,0.97426,0.87975,0.87224,0.86722,0.97447,0.88112,0.87410,64295.33333,4.11398,3280.68837,1405.69377,623.35271



== Tamanho do efeito: amplitude (maior média − menor média) comparada com o ruído ==


hiperparametro,C,gamma (× referência)
métrica,,
ROC-AUC (CV),0.00298,0.00368
MCC (CV),0.00739,0.00969
F1 macro (CV),0.00658,0.00858
acurácia balanceada (CV),0.00561,0.00725
ROC-AUC (validação),0.00357,0.00352
MCC (validação),0.00752,0.00994
F1 macro (validação),0.00653,0.00882
vetores de suporte,6412.00000,4837.66667
tamanho do modelo (MiB),0.51314,0.30047


,hiperparametro,métrica,amplitude,ruído (erro-padrão),amplitude / ruído,leitura
0,C,ROC-AUC (CV),0.0030,0.0002,16.5921,ACIMA do ruído
1,C,MCC (CV),0.0074,0.0006,12.9696,ACIMA do ruído
12,gamma (× referência),ROC-AUC (CV),0.0037,0.0002,20.5077,ACIMA do ruído
13,gamma (× referência),MCC (CV),0.0097,0.0006,17.0033,ACIMA do ruído



== Um de cada vez, a partir do padrão (C = 1, gamma = 1× o de referência) ==


,muda,de,para,Δ ROC-AUC (CV),Δ MCC (CV),Δ ROC-AUC (CV) em erros-padrão,Δ MCC (CV) em erros-padrão,Δ vetores de suporte,Δ tempo de treino nas dobras (s)
0,gamma (× referência),1.0,0.25,-0.00473,-0.00645,-29.01932,-12.56110,6887,138.79932
1,gamma (× referência),1.0,4.00,0.00023,0.00669,1.40408,13.02911,-2411,303.04940
2,C,1.0,10.00,-0.00276,0.00625,-16.92371,12.16244,-3605,-34.83902
3,C,1.0,100.00,-0.00554,0.00830,-33.97668,16.15870,-5484,66.63505



== Decomposição da variação entre as configurações (fração de cada fonte; cada coluna soma 1) ==


,ROC-AUC (CV) (%),MCC (CV) (%),ROC-AUC (validação) (%),vetores de suporte (%),tempo de treino completo (s) (%)
fonte da variação,,,,,
C,33.4,33.6,43.9,53.8,40.5
gamma (× referência),48.4,58.6,41.2,32.6,49.9
interação C × gamma (resíduo),18.2,7.7,14.8,13.6,9.6



== Leitura automática (ROC-AUC e MCC da validação cruzada) ==
  C                     ROC-AUC (CV)              amplitude 0.00298 (16.6× o ruído) → ACIMA do ruído; melhor valor na média: 1
  C                     MCC (CV)                  amplitude 0.00739 (13.0× o ruído) → ACIMA do ruído; melhor valor na média: 100
  gamma (× referência)  ROC-AUC (CV)              amplitude 0.00368 (20.5× o ruído) → ACIMA do ruído; melhor valor na média: 4
  gamma (× referência)  MCC (CV)                  amplitude 0.00969 (17.0× o ruído) → ACIMA do ruído; melhor valor na média: 4

✔ gráficos salvos: influencia_efeitos_principais.png, influencia_interacao.png


## 7. Recall por classe
O recall de cada classe na validação cruzada e na validação separada, para a combinação **escolhida** e para o **padrão**. Não decide a escolha, mas mostra se alguma classe pequena piorou.

In [12]:
# ---------- 7.1 Recall por classe ----------
tab_cls_grade = df_cls.groupby(CHAVE + ["fonte", "classe"])["recall"].mean().reset_index()
def _rt(C, f): return tab_cls_grade[(tab_cls_grade.C == C) & (tab_cls_grade.gamma_fator == f)].pivot(index="classe", columns="fonte", values="recall").reindex(CLASSES)
rt_esc = _rt(C_ESC, F_ESC); out = pd.DataFrame({"n no ajuste": tab_classes["ajuste"], "n na validação": tab_classes["validação"]})
out["escolhida: CV"] = rt_esc["cv_fora_da_dobra"]
if "validacao" in rt_esc: out["escolhida: validação"] = rt_esc["validacao"]
if len(pd_):
    rt_pad = _rt(PADRAO["C"], PADRAO["gamma_fator"]); out["padrão: CV"] = rt_pad["cv_fora_da_dobra"]
    if "validacao" in rt_pad: out["padrão: validação"] = rt_pad["validacao"]; out["Δ validação (escolhida − padrão)"] = out["escolhida: validação"] - out["padrão: validação"]
print("== Recall por classe (a classe prevista é a de maior escore) ==")
display(out.round(5))
col_pior = "validacao" if "validacao" in rt_esc else "cv_fora_da_dobra"; pior_cls = rt_esc[col_pior].idxmin()
print(f"Classe com menor recall na {'validação' if col_pior == 'validacao' else 'validação cruzada'} (escolhida): {pior_cls} ({rt_esc.loc[pior_cls, col_pior]:.4f}; {int(tab_classes.loc[pior_cls, 'validação'])} linhas na validação)")
if "Δ validação (escolhida − padrão)" in out: print(f"Maior queda de recall em relação ao padrão (validação): {out['Δ validação (escolhida − padrão)'].min():+.4f} ({out['Δ validação (escolhida − padrão)'].idxmin()}) | maior ganho: {out['Δ validação (escolhida − padrão)'].max():+.4f} ({out['Δ validação (escolhida − padrão)'].idxmax()})")


== Recall por classe (a classe prevista é a de maior escore) ==


,n no ajuste,n na validação,escolhida: CV,escolhida: validação,padrão: CV,padrão: validação,Δ validação (escolhida − padrão)
BENIGN,40000,10000,0.99738,0.99830,0.99525,0.99590,0.00240
LDAP,24079,6020,0.95444,0.95598,0.94713,0.95066,0.00532
MSSQL,40000,10000,0.92142,0.91790,0.90650,0.89840,0.01950
NetBIOS,15592,3898,0.53572,0.54413,0.52411,0.53797,0.00616
Syn,40000,10000,0.99270,0.99120,0.97435,0.97290,0.01830
UDP,40000,10000,0.98895,0.98840,0.98325,0.98160,0.00680
UDPLag,40000,10000,0.69292,0.70140,0.69535,0.70330,-0.00190


Classe com menor recall na validação (escolhida): NetBIOS (0.5441; 3898 linhas na validação)
Maior queda de recall em relação ao padrão (validação): -0.0019 (UDPLag) | maior ganho: +0.0195 (MSSQL)


## 8. A combinação escolhida: modelo do ajuste e avaliação na validação
O modelo treinado no ajuste inteiro (o treino completo da busca) e as mesmas tabelas do RF: métricas na validação (sem pesos e na proporção do treino), por classe, matriz de confusão, dispersão entre 5 partições e a validação cruzada da escolhida.

In [13]:
# ---------- 8.1 Modelo do ajuste da combinação escolhida ----------
print("Combinação escolhida:", {k: HIPERPARAMETROS_FINAL[k] for k in ("C", "gamma", "gamma_fator", "class_weight")})
sc_val, meta_aj = garantir_completo(C_ESC, F_ESC)                                    # o treino completo (ajuste inteiro → validação) feito na busca
modelo_aj = joblib.load(PASTA / "modelos" / f"{NOME_ESC}.joblib"); modelo_aj.colunas_modelo_ = list(CM)
assert list(modelo_aj.named_steps["svm"].classes_) == list(range(K)), "o modelo não aprendeu todas as classes (ou a ordem delas mudou)"
fit_ajuste = {"relogio_s": meta_aj["tempo_treino_s"], "cpu_s": meta_aj["cpu_s"], "pico_ram_mib": meta_aj["pico_ram_mib"], "acrescimo_ram_mib": meta_aj["acrescimo_ram_mib"]}
print(f"✔ SVM treinado no ajuste ({meta_aj['n_treino']:,} linhas): {fit_ajuste['relogio_s']:.1f} s de relógio, {fit_ajuste['cpu_s']:.1f} s de CPU, pico de RAM {fit_ajuste['pico_ram_mib']:,.0f} MiB (+{fit_ajuste['acrescimo_ram_mib']:,.0f} MiB) | {meta_aj['n_sv']:,} vetores de suporte")
PESO_CLASSE_AJ = {CLASSES[k]: float(len(y_aj) / (K * (y_aj == k).sum())) for k in range(K)}
print("Pesos de classe do ajuste (balanced):", {c: round(v, 3) for c, v in PESO_CLASSE_AJ.items()})

# ---------- 8.2 Métricas na validação ----------
proba_va = sc_val.astype(np.float64)                                                # os mesmos escores (float32) que a busca gravou
m_sem, arr_sem = metricas_multi(y_va, proba_va, None); m_prop, arr_prop = metricas_multi(y_va, proba_va, w_va)
DEPENDE = {k: "sim" for k in ("acuracia", "precisao_macro", "f1_macro", "f1_ponderado", "mcc")}; DEPENDE["roc_auc_ovr_macro"] = "pouco"
ordem = ["roc_auc_ovr_macro", "acuracia_balanceada", "acuracia", "precisao_macro", "f1_macro", "f1_ponderado", "mcc"]
tab_val = pd.DataFrame({"sem pesos": {k: m_sem[k] for k in ordem}, "proporção do treino": {k: m_prop[k] for k in ordem}}); tab_val["depende da proporção?"] = [DEPENDE.get(k, "não") for k in tab_val.index]
print("\n== Métricas na validação (previsão = classe de maior escore; o escore é a decisão um-contra-resto do SVM) ==  (a coluna 'proporção do treino' usa o peso de cada linha da validação)")
display(tab_val.round(5))
tab_cls = pd.DataFrame({"suporte (validação)": arr_sem["suporte"].astype(int), "recall": arr_sem["recall"], "precisão (sem pesos)": arr_sem["precisao"], "F1 (sem pesos)": arr_sem["f1"], "precisão (prop. treino)": arr_prop["precisao"], "F1 (prop. treino)": arr_prop["f1"],
                        "ROC-AUC um-contra-resto": arr_sem["auc"], "peso de classe no ajuste": [PESO_CLASSE_AJ[c] for c in CLASSES]}, index=CLASSES)
print("\n== Por classe (recall e ROC-AUC não dependem da proporção) =="); display(tab_cls.round(4))
cm_val = pd.DataFrame(arr_sem["cm"].astype(int), index=[f"real {c}" for c in CLASSES], columns=[f"prev. {c}" for c in CLASSES])
print("\n== Matriz de confusão na validação (linhas = classe real; colunas = classe prevista) =="); display(cm_val)
cm_rec = pd.DataFrame(arr_sem["cm"] / arr_sem["cm"].sum(axis=1, keepdims=True), index=cm_val.index, columns=cm_val.columns)
print("== Mesma matriz, em fração da classe real (a diagonal é o recall) =="); display(cm_rec.round(4))
fora = [(CLASSES[i], CLASSES[j], int(arr_sem["cm"][i, j]), arr_sem["cm"][i, j] / arr_sem["cm"][i].sum()) for i in range(K) for j in range(K) if i != j and arr_sem["cm"][i, j] > 0]
fora = sorted(fora, key=lambda x: -x[2])[:8]
print("Confusões mais frequentes (real → previsto): " + " | ".join(f"{a}→{b}: {n:,} ({100 * f:.2f}% de {a})" for a, b, n, f in fora))

# ---------- 8.3 Dispersão entre partições da validação ----------
linhas_part = []
for k, (_, idx) in enumerate(StratifiedKFold(K_PARTICOES_VAL, shuffle=True, random_state=SEMENTE).split(X_va, y_va), 1):
    e1, _a = metricas_multi(y_va[idx], proba_va[idx], None); e2, _b = metricas_multi(y_va[idx], proba_va[idx], w_va[idx])
    linhas_part.append({"particao": k, "n": len(idx), "roc_auc_ovr_macro": e1["roc_auc_ovr_macro"], "acuracia_balanceada": e1["acuracia_balanceada"], "acuracia": e1["acuracia"], "f1_macro": e1["f1_macro"], "mcc": e1["mcc"], "f1_macro_prop": e2["f1_macro"], "mcc_prop": e2["mcc"]})
tab_part = pd.DataFrame(linhas_part); resumo_part = tab_part.drop(columns=["particao", "n"]).agg(["mean", "std", "min", "max"]).T
print(f"\n== Dispersão entre {K_PARTICOES_VAL} partições da validação =="); display(tab_part.round(5)); display(resumo_part.round(5))

# ---------- 8.4 Validação cruzada (da busca) para a combinação escolhida ----------
sel_cv = df_fold[(df_fold.C == C_ESC) & (df_fold.gamma_fator == F_ESC)]
tab_cv = sel_cv[["dobra", "roc_auc", "acuracia_balanceada", "acuracia", "f1_macro", "mcc", "tempo_treino_s", "n_sv"]].reset_index(drop=True)
resumo_cv = tab_cv.drop(columns=["dobra"]).agg(["mean", "std", "min", "max"]).T
tab_cv_cls = rt_esc[["cv_fora_da_dobra"]].T.rename(index={"cv_fora_da_dobra": "média das dobras"}).reset_index(names="origem")
print(f"\n== Validação cruzada da combinação escolhida ({K_CV} dobras; as mesmas do RF) ==")
display(tab_cv.round(5)); display(resumo_cv.round(5))
print("Recall por classe na validação cruzada:"); display(tab_cv_cls.round(4))


Combinação escolhida: {'C': 100.0, 'gamma': 0.09514619264617266, 'gamma_fator': 4.0, 'class_weight': 'balanced'}
✔ SVM treinado no ajuste (239,671 linhas): 2730.1 s de relógio, 2723.6 s de CPU, pico de RAM 757 MiB (+429 MiB) | 62,797 vetores de suporte
Pesos de classe do ajuste (balanced): {'BENIGN': 0.856, 'LDAP': 1.422, 'MSSQL': 0.856, 'NetBIOS': 2.196, 'Syn': 0.856, 'UDP': 0.856, 'UDPLag': 0.856}

== Métricas na validação (previsão = classe de maior escore; o escore é a decisão um-contra-resto do SVM) ==  (a coluna 'proporção do treino' usa o peso de cada linha da validação)


,sem pesos,proporção do treino,depende da proporção?
roc_auc_ovr_macro,0.97355,0.96889,pouco
acuracia_balanceada,0.87104,0.87104,não
acuracia,0.89869,0.95838,sim
precisao_macro,0.90105,0.86153,sim
f1_macro,0.87566,0.85978,sim
f1_ponderado,0.89385,0.95783,sim
mcc,0.88245,0.92193,sim



== Por classe (recall e ROC-AUC não dependem da proporção) ==


,suporte (validação),recall,precisão (sem pesos),F1 (sem pesos),precisão (prop. treino),F1 (prop. treino),ROC-AUC um-contra-resto,peso de classe no ajuste
BENIGN,10000,0.9983,0.9897,0.9940,0.9766,0.9873,0.9995,0.8560
LDAP,6020,0.9560,0.8878,0.9207,0.6801,0.7947,0.9873,1.4219
MSSQL,10000,0.9179,0.8263,0.8697,0.9201,0.9190,0.9748,0.8560
NetBIOS,3898,0.5441,0.8636,0.6676,0.5381,0.5411,0.9213,2.1959
Syn,10000,0.9912,0.9344,0.9620,0.9616,0.9762,0.9978,0.8560
UDP,10000,0.9884,0.8192,0.8959,0.9820,0.9852,0.9813,0.8560
UDPLag,10000,0.7014,0.9864,0.8198,0.9722,0.8149,0.9530,0.8560



== Matriz de confusão na validação (linhas = classe real; colunas = classe prevista) ==


,prev. BENIGN,prev. LDAP,prev. MSSQL,prev. NetBIOS,prev. Syn,prev. UDP,prev. UDPLag
real BENIGN,9983,0,0,16,0,0,1
real LDAP,13,5755,207,33,9,0,3
real MSSQL,9,614,9179,196,0,1,1
real NetBIOS,50,98,1614,2121,0,1,14
real Syn,5,1,1,8,9912,0,73
real UDP,4,2,65,40,0,9884,5
real UDPLag,23,12,43,42,687,2179,7014


... (6 outputs left out)

,dobra,roc_auc,acuracia_balanceada,acuracia,f1_macro,mcc,tempo_treino_s,n_sv
0,1,0.97367,0.86832,0.89678,0.87297,0.88043,1503.69665,50604
1,2,0.97320,0.86745,0.89740,0.87211,0.88109,1554.84933,50593
2,3,0.97339,0.86967,0.89717,0.87428,0.88085,1520.81415,50833
3,4,0.97358,0.87157,0.89892,0.87628,0.88281,1539.38164,50857
4,5,0.97288,0.86837,0.89650,0.87302,0.88007,1540.72630,50521


,mean,std,min,max
roc_auc,0.97334,0.00032,0.97288,0.97367
acuracia_balanceada,0.86908,0.00160,0.86745,0.87157
acuracia,0.89736,0.00094,0.89650,0.89892
f1_macro,0.87373,0.00162,0.87211,0.87628
mcc,0.88105,0.00106,0.88007,0.88281
tempo_treino_s,1531.89361,19.86753,1503.69665,1554.84933
n_sv,50681.60000,152.76714,50521.00000,50857.00000


Recall por classe na validação cruzada:


classe,origem,BENIGN,LDAP,MSSQL,NetBIOS,Syn,UDP,UDPLag
0,média das dobras,0.9974,0.9544,0.9214,0.5357,0.9927,0.989,0.6929


## 9. Custo computacional do modelo do ajuste
Tamanho, vetores de suporte, **latência** de 1 fluxo e **vazão** em lote (com 1 processo e, **só se a memória permitir**, com mais).

In [14]:
# ---------- 9.1 Custo do modelo do ajuste ----------
shutil.copyfile(PASTA / "modelos" / f"{NOME_ESC}.joblib", PASTA / "modelo_ajuste.joblib")                      # o modelo do ajuste (já gravado pelo processo de treino) com o nome usado nos outros notebooks
tam_aj = tamanho_do_arquivo(modelo_aj, PASTA / "modelo_ajuste.joblib")
K_PREV = processos_seguros(PROCESSOS_PREVISAO, RAM_PREVISAO_POR_PROCESSO_GIB)
print(f"Previsão em lote com {K_PREV} processo(s) (desejado {PROCESSOS_PREVISAO}; memória livre {ram_livre_gib():.1f} GiB; cada processo precisa de ~{RAM_PREVISAO_POR_PROCESSO_GIB:.1f} GiB × 1,3)")
inf_aj = medir_inferencia_svm(modelo_aj, prep, X_va, X_va_raw, n_lat=N_LATENCIA, semente=SEMENTE, n_lote=N_LOTE_VAZAO, repeticoes_lote=REPETICOES_LOTE, k_par=K_PREV)
print(f"Modelo do ajuste: {tam_aj['arquivo_mib']:.2f} MiB em disco | {tam_aj['n_vetores_suporte']:,} vetores de suporte ({tam_aj['n_sv_por_classe']} por classe) | {tam_aj['n_atributos']} atributos | C = {tam_aj['C']:g}, gamma = {tam_aj['gamma']:.5f}")
display(pd.DataFrame(inf_aj).set_index("n_jobs").T.round(3))


Previsão em lote com 1 processo(s) (desejado 2; memória livre 1.2 GiB; cada processo precisa de ~0.5 GiB × 1,3)
Modelo do ajuste: 4.00 MiB em disco | 62,797 vetores de suporte ([2928, 3227, 21978, 8239, 3533, 9894, 12998] por classe) | 44 atributos | C = 100, gamma = 0.09515


n_jobs,1
n_latencia,1000.000
latencia_modelo_p50_ms,12.556
latencia_modelo_p95_ms,19.968
latencia_modelo_p99_ms,21.359
latencia_fim_a_fim_p50_ms,12.687
latencia_fim_a_fim_p95_ms,20.206
latencia_fim_a_fim_p99_ms,22.873
vazao_modelo_fluxos_s,96.791
vazao_fim_a_fim_fluxos_s,96.548
n_lote,5000.000


## 10. Refit (modelo final)
Refaz a combinação escolhida com **todas as linhas do refit** (as mesmas do RF; ajuste + validação), cronometrando `REPETICOES_TEMPO` vezes, e o **padrão** (C = 1, gamma = 1×) para a comparação no teste. O primeiro refit é o **modelo final**, gravado para o notebook 12m.

In [15]:
# ---------- 10.1 Refit ----------
print(f"Refit: {len(X_rf):,} linhas (as mesmas do notebook 05m e do RF): " + ", ".join(f"{c} {n:,}" for c, n in conta(y_rf).items()))
fazer_padrao = COMPARAR_COM_PADRAO and not ESCOLHEU_PADRAO
barra = Barra(REPETICOES_TEMPO + (1 if fazer_padrao else 0), "Refit do SVM"); reps, p_ref = [], None
for k in range(1, REPETICOES_TEMPO + 1):
    rotulo = f"refit {k}/{REPETICOES_TEMPO}"; barra.avancar(0, rotulo + "...")
    caminho = str(PASTA / "modelo_final_refit.joblib") if k == 1 else None
    out_ = executar_com_limite(_worker_arrays, (X_rf, y_rf, X_va[:5000], C_ESC, GAMAS[F_ESC], caminho, None), LIMITE_FIT_S, barra, rotulo)
    assert out_ is not None, f"o refit passou do limite de tempo ({fmt_tempo(LIMITE_FIT_S)}): aumente LIMITE_FIT_S"
    meta, sc = out_; reps.append({"repeticao": k, "relogio_s": meta["tempo_treino_s"], "cpu_s": meta["cpu_s"], "pico_ram_mib": meta["pico_ram_mib"], "acrescimo_ram_mib": meta["acrescimo_ram_mib"], "n_sv": meta["n_sv"]})
    if k == 1: p_ref = sc
    else: assert meta["n_sv"] == reps[0]["n_sv"] and np.allclose(sc, p_ref, atol=1e-5), "as repetições do refit deram modelos diferentes (o SVM deveria ser determinístico)"
    barra.avancar(1, f"{rotulo}: {fmt_tempo(meta['tempo_treino_s'])}, {meta['n_sv']:,} vetores de suporte")
tab_reps = pd.DataFrame(reps); display(tab_reps.round(3))
modelo_final = joblib.load(PASTA / "modelo_final_refit.joblib"); modelo_final.colunas_modelo_ = list(CM)
fit_refit = {"repeticoes": REPETICOES_TEMPO, "relogio_mediana_s": float(tab_reps.relogio_s.median()), "relogio_min_s": float(tab_reps.relogio_s.min()), "relogio_max_s": float(tab_reps.relogio_s.max()),
             "cpu_mediana_s": float(tab_reps.cpu_s.median()), "pico_ram_max_mib": float(tab_reps.pico_ram_mib.max()), "acrescimo_ram_max_mib": float(tab_reps.acrescimo_ram_mib.max())}
tam_fin = tamanho_do_arquivo(modelo_final, PASTA / "modelo_final_refit.joblib")
modelo_pad_final, fit_refit_pad, tam_pad = None, None, None
if fazer_padrao:
    rotulo = "refit do padrão"; barra.avancar(0, rotulo + "...")
    out_ = executar_com_limite(_worker_arrays, (X_rf, y_rf, X_va[:5000], PADRAO["C"], GAMAS[PADRAO["gamma_fator"]], str(PASTA / "modelo_padrao_refit.joblib"), None), LIMITE_FIT_S, barra, rotulo); assert out_ is not None, "o refit do padrão passou do limite de tempo"
    meta_p, _ = out_; modelo_pad_final = joblib.load(PASTA / "modelo_padrao_refit.joblib"); modelo_pad_final.colunas_modelo_ = list(CM); tam_pad = tamanho_do_arquivo(modelo_pad_final, PASTA / "modelo_padrao_refit.joblib")
    fit_refit_pad = {"relogio_s": meta_p["tempo_treino_s"], "cpu_s": meta_p["cpu_s"], "pico_ram_mib": meta_p["pico_ram_mib"], "n_sv": meta_p["n_sv"]}; barra.avancar(1, f"{rotulo}: {fmt_tempo(meta_p['tempo_treino_s'])}, {meta_p['n_sv']:,} vetores de suporte")
barra.fechar()
inf_fin = medir_inferencia_svm(modelo_final, prep, X_va, X_va_raw, n_lat=N_LATENCIA, semente=SEMENTE, n_lote=N_LOTE_VAZAO, repeticoes_lote=REPETICOES_LOTE, k_par=K_PREV)
print(f"\n✔ Refit: {fit_refit['relogio_mediana_s']:.1f} s (mediana de {REPETICOES_TEMPO}; mín {fit_refit['relogio_min_s']:.1f}, máx {fit_refit['relogio_max_s']:.1f}), CPU {fit_refit['cpu_mediana_s']:.1f} s, pico de RAM {fit_refit['pico_ram_max_mib']:,.0f} MiB (+{fit_refit['acrescimo_ram_max_mib']:,.0f} MiB)")
print(f"Modelo final: {tam_fin['arquivo_mib']:.2f} MiB em disco | {tam_fin['n_vetores_suporte']:,} vetores de suporte ({tam_fin['n_sv_por_classe']} por classe)")
if tam_pad: print(f"Modelo padrão (refit): {tam_pad['arquivo_mib']:.2f} MiB | {tam_pad['n_vetores_suporte']:,} vetores de suporte | treino {fmt_tempo(fit_refit_pad['relogio_s'])}")
print("(latência e vazão do modelo final: medidas na validação só para VELOCIDADE; não são métricas de qualidade)")
display(pd.DataFrame(inf_fin).set_index("n_jobs").T.round(3))


Refit: 299,589 linhas (as mesmas do notebook 05m e do RF): BENIGN 50,000, LDAP 30,099, MSSQL 50,000, NetBIOS 19,490, Syn 50,000, UDP 50,000, UDPLag 50,000


,repeticao,relogio_s,cpu_s,pico_ram_mib,acrescimo_ram_mib,n_sv
0,1,5308.427,5300.547,742.535,493.008,77699


Refit do SVM: concluído em 2h 41min

✔ Refit: 5308.4 s (mediana de 1; mín 5308.4, máx 5308.4), CPU 5300.5 s, pico de RAM 743 MiB (+493 MiB)
Modelo final: 4.88 MiB em disco | 77,699 vetores de suporte ([3331, 4022, 27393, 10217, 4368, 12225, 16143] por classe)
Modelo padrão (refit): 4.99 MiB | 84,556 vetores de suporte | treino 1h 10min
(latência e vazão do modelo final: medidas na validação só para VELOCIDADE; não são métricas de qualidade)


n_jobs,1
n_latencia,1000.000
latencia_modelo_p50_ms,14.523
latencia_modelo_p95_ms,18.035
latencia_modelo_p99_ms,24.320
latencia_fim_a_fim_p50_ms,14.642
latencia_fim_a_fim_p95_ms,18.786
latencia_fim_a_fim_p99_ms,25.004
vazao_modelo_fluxos_s,79.342
vazao_fim_a_fim_fluxos_s,77.945
n_lote,5000.000


## 11. Gravação
Grava os resultados de toda a grade, a escolha, as métricas e o custo do modelo final, a configuração (com as classes, os pesos e a **impressão digital das dobras**) e uma linha em `registro_buscas.csv`.

In [16]:
# ---------- 11.1 Gravação ----------
hw = {"so": platform.platform(), "python": sys.version.split()[0], "cpu": platform.processor(), "nucleos_logicos": os.cpu_count()}
if psutil: hw.update({"nucleos_fisicos": psutil.cpu_count(logical=False), "ram_total_gib": psutil.virtual_memory().total / 2 ** 30})
sw = {"scikit-learn": sklearn.__version__, "numpy": np.__version__, "pandas": pd.__version__, "joblib": joblib.__version__, "psutil": None if psutil is None else psutil.__version__}
protocolo = {"k_particoes_validacao": K_PARTICOES_VAL, "k_cv_ajuste": K_CV, "repeticoes_tempo_refit": REPETICOES_TEMPO, "n_latencia": N_LATENCIA, "processos_max": PROCESSOS_MAX, "ram_por_processo_gib_final": STATS_BUSCA.get("ram_proc_final", RAM_POR_PROCESSO_GIB), "processos_previsao_usados": K_PREV}
PESO_CLASSE_RF = {CLASSES[k]: float(len(y_rf) / (K * (y_rf == k).sum())) for k in range(K)}
config = {"run_id": RUN_ID, "data": datetime.now().isoformat(timespec="seconds"), "modelo": "SVC multiclasse (kernel RBF; 21 SVMs de pares)", "variante": "sem_auc_ajustado", "origem": "notebook 11m (busca de hiperparâmetros do SVM)",
          "hiperparametros": HIPERPARAMETROS_FINAL, "padrao": {"C": PADRAO["C"], "gamma": GAMAS[PADRAO["gamma_fator"]], "gamma_fator": PADRAO["gamma_fator"]}, "semente": SEMENTE, "previsao": "classe de maior escore (decision_function um-contra-resto)",
          "versao_selecao_notebook04": versao_sel, "versao_views_notebook03": versao_views, "classes": {"nomes": CLASSES, "codigos": COD}, "pesos_classe_balanced_ajuste": PESO_CLASSE_AJ, "pesos_classe_balanced_refit": PESO_CLASSE_RF,
          "dados": {"ajuste": len(X_aj), "validacao": len(X_va), "refit": len(X_rf), "linhas_treino_por_classe": N_TREINO, "ajuste_por_classe": N_AJ_C, "validacao_por_classe": N_VA_C, "usadas_por_classe": {c: N_AJ_C[c] + N_VA_C[c] for c in CLASSES}},
          "protocolo": protocolo, "software": sw, "hardware": hw}
salvar_json(config, PASTA / "config_treino.json"); salvar_json(prep.para_dict(), PASTA / "preprocessamento.json"); salvar_json({"classes": CLASSES, "codigos": COD}, PASTA / "classes.json")
_criado_em = json.loads((PASTA / "config_busca.json").read_text(encoding="utf-8")).get("criado_em")
cfg_busca = {"run_id": RUN_ID, "impressao": IMPRESSAO, "impressao_dobras": IMPRESSAO_DOBRAS, "impressao_dobras_rf": RF_IMPRESSAO, "retomada": RETOMANDO, "criado_em": _criado_em, "concluido_em": datetime.now().isoformat(timespec="seconds"),
             "tempo_busca": {"relogio_s": TEMPO_BUSCA_RELOGIO, "sessoes": len(tempos_sessoes), "treino_somado_s": TEMPO_TREINO_SOMADO, "svms": N_SVMS, "detalhe_sessoes": tempos_sessoes.to_dict(orient="records")},
             "grade": GRADE, "k_cv": K_CV, "semente_dobras": SEMENTE, "gamma_referencia": GAMMA0, "class_weight": "balanced", "paralelismo": {"processos_max": PROCESSOS_MAX, "ram_reserva_gib": RAM_RESERVA_GIB}, "classes": CLASSES, "software": sw, "hardware": hw,
             "regra": {"metricas": METRICAS_DECISAO, "mult_erro_padrao": MULT_ERRO_PADRAO, "desempate": "menos vetores de suporte, depois menos tempo de treino, depois menor C"}}
salvar_json(cfg_busca, PASTA / "config_busca.json")
salvar_json({"escolhida": {"C": C_ESC, "gamma_fator": F_ESC, "gamma": GAMAS[F_ESC], "config": ESC["config"]}, "usou_plano_b": bool(FALLBACK), "passo1": INFO_SEL, "n_elegiveis": len(CAND), "elegiveis": CAND[["config", "cv_roc_auc", "cv_mcc", "n_sv"]].to_dict(orient="records"),
             "hiperparametros_finais": HIPERPARAMETROS_FINAL}, PASTA / "selecao_hiperparametros.json")
df_cfg.to_csv(PASTA / "resultados_grade.csv", index=False); df_fold.to_csv(PASTA / "cv_dobras.csv", index=False); df_val.to_csv(PASTA / "validacao_por_config.csv", index=False)
tab_cls_grade.to_csv(PASTA / "recall_por_classe_grade.csv", index=False); tab_comp.to_csv(PASTA / "comparacao_escolhida_padrao.csv", index=False); out.reset_index(names="classe").to_csv(PASTA / "recall_por_classe_escolhida_padrao.csv", index=False)
tab_ef.to_csv(PASTA / "efeitos_principais.csv", index=False); tab_amp.to_csv(PASTA / "amplitude_efeitos.csv", index=False); decomp.reset_index().to_csv(PASTA / "decomposicao_variacao.csv", index=False)
if len(tab_um): tab_um.to_csv(PASTA / "efeito_um_de_cada_vez.csv", index=False)
met_val = {"sem_pesos": m_sem, "proporcao_treino": m_prop, "por_classe": tab_cls.to_dict(orient="index"), "matriz_confusao": cm_val.to_dict(orient="index"),
           "dispersao_particoes": {"k": K_PARTICOES_VAL, "resumo": resumo_part.to_dict(orient="index")}, "cv_ajuste": {"k": K_CV, "origem": "busca (as mesmas dobras do RF)", "resumo": resumo_cv.to_dict(orient="index")}}
salvar_json(met_val, PASTA / "metricas_validacao.json")
tab_val.reset_index(names="metrica").to_csv(PASTA / "metricas_validacao.csv", index=False); tab_cls.reset_index(names="classe").to_csv(PASTA / "metricas_por_classe_validacao.csv", index=False)
cm_val.reset_index(names="classe_real").to_csv(PASTA / "matriz_confusao_validacao.csv", index=False); tab_part.to_csv(PASTA / "dispersao_particoes_validacao.csv", index=False)
tab_cv.to_csv(PASTA / "cv_ajuste.csv", index=False); tab_cv_cls.to_csv(PASTA / "cv_recall_por_classe.csv", index=False)
custo = {"fit_ajuste": fit_ajuste, "fit_refit": fit_refit, "fit_refit_repeticoes": reps, "modelo_ajuste": tam_aj, "modelo_final": tam_fin, "inferencia_modelo_ajuste": inf_aj, "inferencia_modelo_final": inf_fin, "modelo_padrao": tam_pad, "fit_refit_padrao": fit_refit_pad}
salvar_json(custo, PASTA / "custo_computacional.json")
linhas_custo_csv = [{"item": "fit_ajuste", **fit_ajuste}, {"item": "fit_refit", **fit_refit}] + [{"item": f"inferencia_final_njobs{r['n_jobs']}", **r} for r in inf_fin] + [{"item": "modelo_final", **tam_fin}]
pd.DataFrame(linhas_custo_csv).to_csv(PASTA / "custo_computacional.csv", index=False)
rf_df[["id_linha"]].assign(y=y_rf).to_csv(PASTA / "refit_ids.csv", index=False)
i1 = next(r for r in inf_fin if r["n_jobs"] == 1); iN = inf_fin[-1]
registro = {"run_id": RUN_ID, "data": config["data"], "retomada": RETOMANDO, "n_configuracoes": len(CONFIGS), "k_cv": K_CV, "escolhida_C": C_ESC, "escolhida_gamma_fator": F_ESC, "tempo_busca_relogio_s": TEMPO_BUSCA_RELOGIO, "tempo_treino_somado_s": TEMPO_TREINO_SOMADO,
            "n_sessoes": len(tempos_sessoes), "usou_plano_b": bool(FALLBACK), "n_elegiveis": len(CAND), "cv_roc_auc": ESC["cv_roc_auc"], "cv_roc_auc_ep": ESC["cv_roc_auc_ep"], "cv_mcc": ESC["cv_mcc"], "cv_mcc_ep": ESC["cv_mcc_ep"], "cv_f1_macro": ESC["cv_f1_macro"],
            "roc_auc_val": m_sem["roc_auc_ovr_macro"], "acuracia_balanceada_val": m_sem["acuracia_balanceada"], "f1_macro_val": m_sem["f1_macro"], "mcc_val": m_sem["mcc"], "f1_macro_prop_val": m_prop["f1_macro"], "mcc_prop_val": m_prop["mcc"],
            "fit_refit_relogio_mediana_s": fit_refit["relogio_mediana_s"], "modelo_final_mib": tam_fin["arquivo_mib"], "vetores_de_suporte": tam_fin["n_vetores_suporte"], "latencia_p50_ms_njobs1": i1["latencia_fim_a_fim_p50_ms"],
            "vazao_fluxos_s_njobs1": i1["vazao_fim_a_fim_fluxos_s"], "vazao_fluxos_s_njobs_orig": iN["vazao_fim_a_fim_fluxos_s"], "pasta": str(PASTA)}
caminho_reg = ARTEFATOS_BASE / "registro_buscas.csv"; pd.DataFrame([registro]).to_csv(caminho_reg, mode="a", header=not caminho_reg.exists(), index=False)
print(f"✔ Artefatos gravados em {PASTA}\n✔ Registro acrescentado a {caminho_reg}")
for f in sorted(PASTA.iterdir()):
    if f.is_file(): print(f"   {f.name:42s} {f.stat().st_size / 2**10:>10,.1f} KiB")


✔ Artefatos gravados em C:\Users\Bruno\downloads\tcc_codigo\railway-ch\artefatos_modelos\svm_multiclasse_busca\svm_busca_20261107_013148
✔ Registro acrescentado a C:\Users\Bruno\downloads\tcc_codigo\railway-ch\artefatos_modelos\svm_multiclasse_busca\registro_buscas.csv
   amplitude_efeitos.csv                             2.2 KiB
   classes.json                                      0.2 KiB
   comparacao_escolhida_padrao.csv                   1.3 KiB
   config_busca.json                                 1.6 KiB
   config_treino.json                                2.9 KiB
   custo_computacional.csv                           1.0 KiB
   custo_computacional.json                          2.6 KiB
   cv_ajuste.csv                                     0.7 KiB
   cv_dobras.csv                                     9.1 KiB
   cv_recall_por_classe.csv                          0.1 KiB
   decomposicao_variacao.csv                         1.0 KiB
   dispersao_particoes_validacao.csv                 0.8 Ki

## 12. Conferências
Confere, de forma independente: que a busca está completa, que as dobras são as do RF, que os modelos gravados reproduzem os escores, que os pesos de classe conferem, que o refit usa as mesmas linhas do notebook 05m e que a escolha é reproduzível. O arquivo `FINALIZADO` só é criado se todas passarem.

In [17]:
# ---------- 12.1 Conferências ----------
from sklearn.utils.class_weight import compute_class_weight
ok = []
def verifica(nome, cond, detalhe=""):
    ok.append(bool(cond)); print(("✔" if cond else "✘"), nome, detalhe)
refit_sel_ids = consultar("SELECT toString(id_linha) AS id_linha FROM " + TAB_REFIT)
verifica("Busca completa: todos os treinos gravados (nenhuma configuração ficou de fora)", len(CFG_OK) == len(CONFIGS) and all(feito(*t) for t in TAREFAS), f"({len(TAREFAS)} treinos, {len(CONFIGS)} configurações)")
verifica("Dobras cobrem cada linha do ajuste exatamente uma vez, com todas as classes", np.array_equal(np.sort(np.concatenate([ite for _, ite in DOBRAS])), np.arange(len(y_aj))) and all(len(np.unique(y_aj[ite])) == K for _, ite in DOBRAS))
verifica("As dobras são as mesmas do RF (impressão digital igual à gravada pelo 07m)" if RF_IMPRESSAO else "As dobras: impressão digital do RF não encontrada (comparação omitida)", (RF_IMPRESSAO == IMPRESSAO_DOBRAS) if RF_IMPRESSAO else True, f"({IMPRESSAO_DOBRAS})")
verifica("O modelo do ajuste gravado reproduz os escores da validação da busca", np.allclose(modelo_aj.decision_function(X_va[:2000]), sc_val[:2000], atol=1e-4))
verifica("O modelo final (refit) gravado reproduz os escores medidos no refit", np.allclose(modelo_final.decision_function(X_va[:2000]), p_ref[:2000], atol=1e-4))
svm_f, svm_a = modelo_final.named_steps["svm"], modelo_aj.named_steps["svm"]
verifica("Hiperparâmetros dos modelos gravados = combinação escolhida (C, gamma, class_weight)", all(abs(m.C - C_ESC) < 1e-12 and abs(m._gamma - GAMAS[F_ESC]) < 1e-12 and m.class_weight == "balanced" for m in (svm_f, svm_a)))
verifica("Os modelos aprenderam as classes 0..K-1, e os atributos estão na ordem do pré-processamento", list(svm_f.classes_) == list(svm_a.classes_) == list(range(K)) and list(modelo_final.colunas_modelo_) == list(modelo_aj.colunas_modelo_) == CM)
prep_lido = json.loads((PASTA / "preprocessamento.json").read_text(encoding="utf-8"))
verifica("Ordem dos atributos e medianas gravadas = as da tabela de auditoria do notebook 03m", prep_lido["colunas_modelo"] == CM and prep_lido["medianas_treino"] == MEDIANAS and set(prep_lido["colunas_retiradas_ablacao"]) == set(RETIRADAS))
verifica("Ajuste e validação com as linhas planejadas por classe (notebook 04m)", conta(y_aj) == N_AJ_C and conta(y_va) == N_VA_C, f"({len(X_aj):,} e {len(X_va):,})")
verifica("Refit = ajuste + validação: todas as linhas selecionadas, por classe, e as mesmas do notebook 05m", conta(y_rf) == {c: N_AJ_C[c] + N_VA_C[c] for c in CLASSES} and set(aj["id_linha"]) | set(va["id_linha"]) == set(rf_df["id_linha"]) == set(refit_sel_ids["id_linha"]), f"({len(X_rf):,} linhas)")
verifica("Pesos de classe (balanced) do SVM conferem com o scikit-learn, no ajuste e no refit", all(abs(svm_a.class_weight_[k] - w) < 1e-12 for k, w in enumerate(compute_class_weight("balanced", classes=np.arange(K), y=y_aj))) and all(abs(svm_f.class_weight_[k] - w) < 1e-12 for k, w in enumerate(compute_class_weight("balanced", classes=np.arange(K), y=y_rf))))
_esc2 = escolher(pd.read_csv(PASTA / "resultados_grade.csv"))[0]
verifica("A escolha é reproduzível a partir do resultados_grade.csv gravado", _esc2["config"] == ESC["config"], f"({ESC['config']})")
verifica("Decomposição da variação: cada coluna soma 100%", bool(np.allclose(decomp.sum(), 1.0, atol=1e-9)) if COMPLETA else True)
verifica("Escores finitos e matrizes de confusão fecham", bool(np.isfinite(sc_val).all()) and int(arr_sem["cm"].sum()) == len(y_va) and abs(arr_prop["cm"].sum() - float(w_va.sum())) < 1e-6)
verifica("Pesos da validação somam as linhas de cada classe no treino", all(abs(float(w_va[y_va == COD[c]].sum()) - N_TREINO[c]) < 1e-6 * N_TREINO[c] for c in CLASSES))
verifica("Métricas que não dependem da proporção são iguais nas duas colunas", abs(m_sem["acuracia_balanceada"] - m_prop["acuracia_balanceada"]) < 1e-9 and np.allclose(arr_sem["recall"], arr_prop["recall"], atol=1e-12))
verifica("Paralelismo: nenhum treino foi interrompido por falta de memória sem ser refeito (todos os treinos concluíram)", all(feito(*t) for t in TAREFAS))
verifica("A impressão digital das dobras foi gravada na configuração da busca", json.loads((PASTA / "config_busca.json").read_text(encoding="utf-8")).get("impressao_dobras") == IMPRESSAO_DOBRAS, f"({IMPRESSAO_DOBRAS})")
verifica("Arquivos esperados gravados", all((PASTA / f).exists() for f in ("modelo_ajuste.joblib", "modelo_final_refit.joblib", "config_treino.json", "config_busca.json", "preprocessamento.json", "classes.json", "metricas_validacao.json", "metricas_validacao.csv",
                                                                       "metricas_por_classe_validacao.csv", "matriz_confusao_validacao.csv", "dispersao_particoes_validacao.csv", "cv_ajuste.csv", "custo_computacional.json", "custo_computacional.csv", "selecao_hiperparametros.json",
                                                                       "tempos_busca.csv", "resultados_grade.csv", "cv_dobras.csv", "validacao_por_config.csv", "recall_por_classe_grade.csv", "efeitos_principais.csv", "decomposicao_variacao.csv", "refit_ids.csv")) and (not fazer_padrao or (PASTA / "modelo_padrao_refit.joblib").exists()))
verifica("Custo medido: tempos positivos", fit_ajuste["relogio_s"] > 0 and fit_refit["relogio_mediana_s"] > 0 and inf_fin[0]["vazao_fim_a_fim_fluxos_s"] > 0)
assert all(ok), "há checagens reprovadas acima"
(PASTA / "FINALIZADO").write_text(datetime.now().isoformat(timespec="seconds"), encoding="utf-8")
print(f"\n✔ {sum(ok)}/{len(ok)} checagens aprovadas.")
print(f"\nPróximo passo: notebook 12m (avaliação no teste, uma vez), apontando para {PASTA.name}")


  conexão instável (ConnectionAbortedError: [WinError 10053] Uma conexão estabelecida foi anulada pelo software no computador host); nova tentativa em 5s...
✔ Busca completa: todos os treinos gravados (nenhuma configuração ficou de fora) (54 treinos, 9 configurações)
✔ Dobras cobrem cada linha do ajuste exatamente uma vez, com todas as classes 
✔ As dobras são as mesmas do RF (impressão digital igual à gravada pelo 07m) (7e97c47f)
✔ O modelo do ajuste gravado reproduz os escores da validação da busca 
✔ O modelo final (refit) gravado reproduz os escores medidos no refit 
✔ Hiperparâmetros dos modelos gravados = combinação escolhida (C, gamma, class_weight) 
✔ Os modelos aprenderam as classes 0..K-1, e os atributos estão na ordem do pré-processamento 
✔ Ordem dos atributos e medianas gravadas = as da tabela de auditoria do notebook 03m 
✔ Ajuste e validação com as linhas planejadas por classe (notebook 04m) (239,671 e 59,918)
✔ Refit = ajuste + validação: todas as linhas selecionadas, p